# Jalon 3 — A, B et un agent malveillant

**Situation.** A envoie à B des objets étiquetés par un modèle MR-Sort. Un agent malveillant ajoute, pour chaque objet de A, un objet étiqueté par **un autre modèle MR-Sort**. B reçoit le mélange, sans savoir qui a produit quoi, et doit retrouver le modèle de A.

Ce notebook avance par sous-tâches :
- **T1** : génération du scénario (cette section) ;
- T2 à T5 : algorithme de B, identification du modèle de A, évaluation, exemple ;
- T6 à T8 : stratégie de brouillage.

## T1. Génération du scénario

`make_scenario` produit d'un côté **ce que voit B** (`X`, `y`), de l'autre **ce que B ne voit pas** et qui sert à l'évaluation : la source de chaque objet, les deux modèles et un jeu de test étiqueté par le modèle de A.

| Paramètre | Défaut |
|---|---|
| critères `n` | 6 |
| catégories `p` | 3 |
| objets de A `n_A` | 300 |
| objets du malveillant | la moitié de ceux de A (`ratio_malveillant = 0.5`, voir T3) |
| loi des performances `x` | uniforme sur $[0,1]^n$ pour A et pour le malveillant |
| modèle du malveillant | tiré avec le même générateur que celui de A |

In [ ]:
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
import lincs, lincs.classification as C

print("lincs", lincs.__version__)

### Règle MR-Sort

Fonctions reprises du jalon 2 (`02_jalon2_final`) : `predict` applique la règle MR-Sort en numpy, en float32 comme lincs, avec des poids bruts et un seuil de majorité égal à 1. Un modèle est un dictionnaire `profiles` (forme $(p-1, n)$), `raw_weights`, `weights` (somme 1) et `lmbda`, avec `raw_weights = weights / lmbda`.

In [ ]:
def concordance(model, X):
    # C[k, h] = somme des poids (bruts) des critères où x_k >= b^h, en float32 comme lincs
    X32 = np.asarray(X, dtype=np.float32)
    W32 = np.asarray(model["raw_weights"], dtype=np.float32)
    return np.stack([(X32 >= b.astype(np.float32)).astype(np.float32) @ W32 for b in model["profiles"]],
                    axis=1).astype(float)

def predict(model, X):
    return np.cumprod(concordance(model, X) >= 1., axis=1).sum(axis=1)       # nb de frontières franchies

### Modèle MR-Sort aléatoire

- **Poids** : Dirichlet(1, …, 1), donc positifs de somme 1.
- **λ** : uniforme dans $[0{,}5 ;\, 1[$. En dessous de 0,5, deux coalitions disjointes pourraient franchir la même frontière.
- **Profils** : pour chaque critère, $p-1$ valeurs tirées dans $[0,1]$ puis triées, ce qui impose $b^1 \le b^2 \le \dots$.
- **Contrôle de qualité** : un modèle tiré au hasard peut être dégénéré (presque tous les objets dans une seule catégorie). On le **rejette** si sa plus petite catégorie contient moins de `min_part_categorie` des objets d'un grand tirage de contrôle (10 000 points). Les deux sources passent par le même filtre : elles restent symétriques.

*Pourquoi rejeter :* avec une catégorie quasi vide, prédire une constante suffit pour être précis, les étiquettes des deux sources se contredisent à peine, et il reste trop peu d'exemples pour apprendre les profils de cette catégorie.

In [ ]:
LOIS_X = {"uniforme": lambda rng, k, n: rng.random((k, n))}       # loi des performances : (rng, nb d'objets, n) -> (k, n)

def draw_model(n, p, rng):
    w = rng.dirichlet(np.ones(n))
    lmbda = rng.uniform(0.5, 1.0)
    return {"profiles": np.sort(rng.random((p - 1, n)), axis=0), "weights": w, "raw_weights": w / lmbda, "lmbda": lmbda}

def category_shares(model, p, loi_x, rng, n_control=10_000):
    X = LOIS_X[loi_x](rng, n_control, model["profiles"].shape[1])
    return np.bincount(predict(model, X), minlength=p) / n_control

def random_model(n, p, rng, loi_x="uniforme", min_part_categorie=0.10, max_essais=1000):
    # modèle MR-Sort aléatoire dont chaque catégorie contient au moins min_part_categorie des objets
    for essai in range(1, max_essais + 1):
        model = draw_model(n, p, rng)
        shares = category_shares(model, p, loi_x, rng)
        if shares.min() >= min_part_categorie:
            return dict(model, shares=shares, essais=essai)
    raise RuntimeError(f"aucun modèle valide en {max_essais} essais (min_part_categorie={min_part_categorie}, "
                       f"n={n}, p={p}) : seuil trop élevé ?")

### Configuration et scénario

Sources aléatoires indépendantes (`SeedSequence`) : le modèle de A, le modèle du malveillant, les $x$ de chaque source, le mélange et le jeu de test ne partagent pas leur générateur. Changer une option ne décale donc pas les autres tirages.

Le jeu de test suit la loi de A et est étiqueté par le modèle de A : c'est la référence de la qualité de B. Les lignes sont **mélangées** pour que la position ne trahisse pas la source.

In [ ]:
@dataclass
class ScenarioConfig:
    n: int = 6                              # critères
    p: int = 3                              # catégories
    n_A: int = 300                          # objets envoyés par A
    ratio_malveillant: float = 0.5          # objets du malveillant = ratio * n_A
    loi_x: str = "uniforme"                 # loi des performances de A (et du jeu de test)
    loi_x_malveillant: str | None = None    # None : même loi que A
    min_part_categorie: float = 0.10        # contrôle de qualité des modèles tirés (0 : désactivé)
    n_test: int = 1000                      # taille du jeu de test étiqueté par A

    @property
    def n_M(self):
        return int(round(self.ratio_malveillant * self.n_A))

@dataclass
class Scenario:
    config: ScenarioConfig
    seed: int
    X: np.ndarray                           # ce que voit B : performances (N, n)
    y: np.ndarray                           # ce que voit B : catégories (N,)
    source: np.ndarray                      # caché à B : 0 = A, 1 = malveillant
    model_A: dict
    model_M: dict
    X_test: np.ndarray                      # jeu de test uniforme, étiqueté par le modèle de A
    y_test: np.ndarray

def make_scenario(config=None, seed=0):
    cfg = config or ScenarioConfig()
    rngs = [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(7)]
    r_mA, r_mM, r_xA, r_xM, r_mix, r_test, _ = rngs
    loi_M = cfg.loi_x_malveillant or cfg.loi_x

    mA = random_model(cfg.n, cfg.p, r_mA, cfg.loi_x, cfg.min_part_categorie)
    mM = random_model(cfg.n, cfg.p, r_mM, loi_M, cfg.min_part_categorie)

    XA = LOIS_X[cfg.loi_x](r_xA, cfg.n_A, cfg.n)
    XM = LOIS_X[loi_M](r_xM, cfg.n_M, cfg.n)
    X = np.vstack([XA, XM])
    y = np.concatenate([predict(mA, XA), predict(mM, XM)])
    source = np.concatenate([np.zeros(cfg.n_A, int), np.ones(cfg.n_M, int)])

    order = r_mix.permutation(len(X))                                    # B ne doit pas voir l'ordre d'émission
    X_test = LOIS_X[cfg.loi_x](r_test, cfg.n_test, cfg.n)
    return Scenario(cfg, seed, X[order], y[order], source[order], mA, mM, X_test, predict(mA, X_test))

### Exemple

Un scénario avec les paramètres par défaut. Le dernier tableau est exactement ce que reçoit B : des performances et une catégorie, sans la source.

In [ ]:
sc = make_scenario(seed=0)
print(f"{len(sc.X)} objets reçus par B : {(sc.source == 0).sum()} de A, {(sc.source == 1).sum()} du malveillant")

parts = pd.DataFrame({
    "A": np.bincount(sc.y[sc.source == 0], minlength=3) / (sc.source == 0).sum(),
    "malveillant": np.bincount(sc.y[sc.source == 1], minlength=3) / (sc.source == 1).sum(),
    "B voit (mélange)": np.bincount(sc.y, minlength=3) / len(sc.y)}, index=[f"catégorie {h}" for h in range(3)])
display(parts.round(3))
print("essais nécessaires pour tirer les modèles : A", sc.model_A["essais"], "| malveillant", sc.model_M["essais"])

vu_par_B = pd.DataFrame(sc.X, columns=[f"c{i + 1}" for i in range(sc.X.shape[1])]).assign(categorie=sc.y)
display(vu_par_B.head(8).round(3))

### Tests de T1

1. reproductibilité (même graine → mêmes données) ;
2. étiquettes cohérentes avec le modèle de la source ;
3. modèles valides (poids, λ, profils croissants) ;
4. équilibre des catégories, contrôle de qualité effectif ;
5. mélange : tailles des sources et ordre mélangé ;
6. indiscernabilité : la loi des $x$ ne dépend pas de la source ;
7. accord entre `predict` (numpy) et lincs sur ces modèles ;
8. le seuil trop élevé lève bien une erreur.

In [ ]:
def same(a, b):
    return all(np.array_equal(getattr(a, f), getattr(b, f)) for f in ("X", "y", "source", "X_test", "y_test"))

def valid_model(m, n, p):
    return (np.isclose(m["weights"].sum(), 1) and (m["weights"] >= 0).all() and 0.5 <= m["lmbda"] < 1
            and m["profiles"].shape == (p - 1, n) and (np.diff(m["profiles"], axis=0) >= 0).all()
            and ((0 <= m["profiles"]) & (m["profiles"] <= 1)).all())

cfg = ScenarioConfig()
seeds = range(30)
scs = [make_scenario(cfg, s) for s in seeds]

# 1. reproductibilité, et graines différentes -> données différentes
assert same(make_scenario(cfg, 5), make_scenario(cfg, 5))
assert not same(make_scenario(cfg, 5), make_scenario(cfg, 6))

for s in scs:
    A, M = s.source == 0, s.source == 1
    # 2. les étiquettes viennent bien du modèle de la source (et le jeu de test, du modèle de A)
    assert (predict(s.model_A, s.X[A]) == s.y[A]).all() and (predict(s.model_M, s.X[M]) == s.y[M]).all()
    assert (predict(s.model_A, s.X_test) == s.y_test).all()
    # 3. modèles valides
    assert valid_model(s.model_A, cfg.n, cfg.p) and valid_model(s.model_M, cfg.n, cfg.p)
    # 4. aucune catégorie sous le seuil (sur le tirage de contrôle du modèle)
    assert s.model_A["shares"].min() >= cfg.min_part_categorie and s.model_M["shares"].min() >= cfg.min_part_categorie
    # 5. tailles des sources, valeurs dans [0, 1], catégories valides
    assert A.sum() == cfg.n_A and M.sum() == cfg.n_M and s.X.shape == (cfg.n_A + cfg.n_M, cfg.n)
    assert s.X.min() >= 0 and s.X.max() <= 1 and set(np.unique(s.y)) <= set(range(cfg.p))
print("tests 1 à 5 : ok sur", len(scs), "scénarios")

# 5 (suite). le mélange est mélangé : la première moitié ne contient pas que A
frac_A_first_half = np.mean([(s.source[:len(s.source) // 2] == 0).mean() for s in scs])
attendu_A = cfg.n_A / (cfg.n_A + cfg.n_M)                             # part de A dans l'ensemble des lignes
print(f"part de A dans la première moitié des lignes : {frac_A_first_half:.3f} (attendu {attendu_A:.3f})")
assert abs(frac_A_first_half - attendu_A) < 0.05

# 6. indiscernabilité : test de Kolmogorov-Smirnov, par critère, A contre malveillant
pvals = np.array([[stats.ks_2samp(s.X[s.source == 0][:, i], s.X[s.source == 1][:, i]).pvalue
                   for i in range(cfg.n)] for s in scs])
print(f"KS : p-valeur moyenne {pvals.mean():.3f} (attendu ~0.5), part de p < 0.05 : {(pvals < 0.05).mean():.3f} (attendu ~0.05)")
assert (pvals < 0.05).mean() < 0.12

# 4 (suite). catégories des modèles tirés, sur tous les scénarios
shares = np.array([s.model_A["shares"] for s in scs] + [s.model_M["shares"] for s in scs])
print("part de catégorie minimale : min", shares.min().round(3), "| moyenne", shares.min(axis=1).mean().round(3))
print("essais de rejet : moyenne", np.mean([s.model_A["essais"] for s in scs]).round(2),
      "| max", max(s.model_A["essais"] for s in scs))

In [ ]:
# 7. accord numpy / lincs sur un modèle tiré au hasard
s = scs[0]
problem = C.Problem([C.Criterion(f"c{i + 1}", C.Criterion.RealValues(C.Criterion.PreferenceDirection.increasing, 0., 1.))
                     for i in range(cfg.n)], [C.Category(f"cat{h}") for h in range(cfg.p)])
m = s.model_A
lincs_model = C.Model(problem,
    [C.AcceptedValues(C.AcceptedValues.RealThresholds([float(m["profiles"][h, i]) for h in range(cfg.p - 1)]))
     for i in range(cfg.n)],
    [C.SufficientCoalitions(C.SufficientCoalitions.Weights([float(w) for w in m["raw_weights"]]))
     for _ in range(cfg.p - 1)])
alts = C.Alternatives(problem, [C.Alternative(f"a{k}", [C.Performance(C.Performance.Real(float(v))) for v in row])
                                for k, row in enumerate(s.X)])
C.classify_alternatives(problem, lincs_model, alts)
accord = np.mean([a.category_index for a in alts.alternatives] == predict(m, s.X))
print("accord numpy / lincs :", accord)
assert accord == 1.0

# 8. un seuil impossible lève une erreur claire
try:
    random_model(6, 3, np.random.default_rng(0), min_part_categorie=0.4, max_essais=50)
    raise SystemExit("erreur attendue")
except RuntimeError as e:
    print("erreur attendue levée :", e)

## T2. Algorithme de B : inv-2-MR-Sort accéléré

B reçoit le mélange `(X, y)` et ne sait ni quelle part vient de A, ni quelle part vient du malveillant. Il applique l'algorithme du jalon 2 (alternance apprentissage lincs par segment / réaffectation, voir `02_jalon2_final`), avec trois changements pour le rendre rapide :

1. **Arrêt sur un seuil de changements** : on s'arrête quand moins de `TOL_CHANGES` (1 %) des objets changent de segment, ou après `MAX_ITER` itérations. Avec `TOL_CHANGES = 0`, on retrouve l'arrêt strict du jalon 2.
2. **Départs en parallèle** (`joblib`, backend `loky`) : les 4 départs (1 guidé + 3 aléatoires) sont indépendants, chacun tourne dans son processus. `loky` sérialise les fonctions du notebook avec cloudpickle.
3. **Modèles purement numpy en sortie** : les objets lincs (`_lincs`, `_problem`) ne sont pas renvoyés par les processus. Ils ne servent qu'à l'apprentissage.

Le partage des objets ambigus (`share_ambiguous`) est abandonné : le jalon 2 a conclu qu'il ne faisait pas mieux (phase 1).

B reste **aveugle** : l'algorithme ne connaît ni la proportion des sources ni le niveau de bruit.

In [ ]:
import re
import time
from pathlib import Path

import yaml
from joblib import Parallel, delayed

ITER_SECONDS   = 1.0     # temps lincs par apprentissage, pendant les itérations (pour 6 critères / 3 catégories)
POLISH_SECONDS = 4.0     # temps lincs de l'apprentissage final (polish), même référence
MAX_ITER       = 10      # nombre maximal d'itérations par départ (20 au jalon 2 ; 10 retenu par la validation ci-dessous)
TOL_CHANGES    = 0.01    # arrêt quand moins de 1 % des objets changent de segment (0 : arrêt strict du jalon 2)
INITS          = ("errors", "random", "random", "random")   # départs : 1 guidé + 3 aléatoires
MIN_FRAC       = 0.10    # taille minimale d'un segment pour apprendre son modèle

In [ ]:
def n_params(n, p):
    return n * (p - 1) + n + 1

def budget(base_seconds, n, p):
    # temps alloué à lincs, proportionnel au nombre de paramètres du modèle (6 critères / 3 catégories = référence)
    return base_seconds * n_params(n, p) / n_params(6, 3)

def make_problem(n, p):
    crits = [C.Criterion(f"c{i + 1}", C.Criterion.RealValues(C.Criterion.PreferenceDirection.increasing, 0., 1.))
             for i in range(n)]
    return C.Problem(crits, [C.Category(f"cat{h}") for h in range(p)])

def to_alts(problem, X, y=None):
    return C.Alternatives(problem, [
        C.Alternative(f"a{k}", [C.Performance(C.Performance.Real(float(v))) for v in X[k]],
                      None if y is None else int(y[k])) for k in range(len(X))])

def learn_mrsort(X, y, p, seed=0, seconds=2.0, models=9, patience=30):
    # heuristique weights-profiles-breed de lincs : un seul modèle MR-Sort
    problem = make_problem(X.shape[1], p)
    pre = C.PreprocessedLearningSet(problem, to_alts(problem, X, y))
    mbl = C.LearnMrsortByWeightsProfilesBreed.ModelsBeingLearned(pre, models, int(seed))
    init = C.InitializeProfilesForProbabilisticMaximalDiscriminationPowerPerCriterion(pre, mbl)
    weights = C.OptimizeWeightsUsingGlop(pre, mbl)
    profiles = C.ImproveProfilesWithAccuracyHeuristicOnCpu(pre, mbl)
    breed = C.ReinitializeLeastAccurate(mbl, init, weights, count=models // 2)
    term = C.TerminateWhenAny([C.TerminateAtAccuracy(mbl, len(X)), C.TerminateAfterSeconds(seconds),
                               C.TerminateAfterIterationsWithoutProgress(mbl, patience)])
    m = C.LearnMrsortByWeightsProfilesBreed(pre, mbl, init, weights, profiles, breed, term).perform()
    B = np.array([[np.inf if av.real_thresholds.thresholds[h] is None else av.real_thresholds.thresholds[h]
                   for av in m.accepted_values] for h in range(p - 1)])          # (p-1, n)
    W = np.array(m.sufficient_coalitions[0].weights.criterion_weights)           # seuil de majorité = 1 (lincs)
    return {"profiles": B, "raw_weights": W, "weights": W / W.sum(), "lmbda": 1. / W.sum()}

def margins(model, X, y):
    # marge de x dans sa catégorie y : positive si bien classé, d'autant plus grande que le classement est solide
    Cn = (concordance(model, X) - 1.) / model["raw_weights"].sum()          # C(x, b^h) - lambda, poids normalisés
    m = np.full(len(X), np.inf)
    low = y >= 1                                                             # doit franchir b^{h-1}
    m[low] = np.minimum(m[low], Cn[low, y[low] - 1] + 1e-12)
    up = np.where(y <= Cn.shape[1] - 1)[0]                                   # ne doit pas franchir b^h
    m[up] = np.minimum(m[up], -Cn[up, y[up]] - 1e-12)
    m[np.isinf(m)] = 1.
    return m

In [ ]:
def scores(models, X, y):
    E = np.stack([np.abs(predict(m, X) - y) for m in models], axis=1)     # (N, 2) écart de catégorie
    M = np.stack([margins(m, X, y) for m in models], axis=1)              # (N, 2) marge
    return E, M

def reassign(models, X, y, seg):
    E, M = scores(models, X, y)
    key = E - 1e-3 * np.clip(M, -1, 1)               # lexicographique : l'écart domine, la marge départage
    best = key.argmin(axis=1)
    rows = np.arange(len(X))
    move = key[rows, best] < key[rows, seg] - 1e-12  # strictement mieux expliqué ailleurs
    return np.where(move, best, seg), E

def fit_segments(X, y, p, seg, rng, seconds):
    models = []
    for k in (0, 1):
        idx = np.where(seg == k)[0]
        size_min = int(max(10, MIN_FRAC * len(X)))
        if len(idx) < size_min:                       # segment trop petit : tirage aléatoire
            idx = rng.choice(len(X), size=size_min, replace=False)
        models.append(learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=seconds))
    return models

def init_partition(X, y, p, how, rng, seconds):
    if how == "random":
        return rng.integers(0, 2, len(X))
    single = learn_mrsort(X, y, p, seed=int(rng.integers(1e9)), seconds=seconds)
    return (predict(single, X) != y).astype(int)      # "errors" : erreurs du modèle unique -> segment 1

def learn_2_mrsort(X, y, p, init="errors", seed=0, max_iter=MAX_ITER, tol_changes=TOL_CHANGES,
                   iter_seconds=ITER_SECONDS, polish_seconds=POLISH_SECONDS):
    # un départ : alternance apprentissage / réaffectation, puis polish. Renvoie le meilleur état rencontré.
    rng = np.random.default_rng(seed)
    n = X.shape[1]
    t_iter, t_polish = budget(iter_seconds, n, p), budget(polish_seconds, n, p)
    seg = init_partition(X, y, p, init, rng, t_iter)
    best, history = None, []

    for it in range(max_iter):
        models = fit_segments(X, y, p, seg, rng, t_iter)                   # apprentissage
        new_seg, E = reassign(models, X, y, seg)                           # réaffectation
        score = (E.min(axis=1) == 0).mean()                                # expliqués par au moins un modèle
        changes = (new_seg != seg).mean()
        sizes = np.bincount(new_seg, minlength=2)
        history.append(dict(iteration=it + 1, expliques=score, petit_segment=sizes.min() / sizes.sum(),
                            ambigus=(E == 0).all(axis=1).mean(), changements=changes))
        if best is None or score > best["score"]:
            best = dict(score=score, models=models, seg=new_seg.copy(), E=E)
        if changes <= tol_changes:                                         # arrêt : partition (presque) stable
            break
        seg = new_seg

    # polish : apprentissage plus long sur la partition retenue
    seg = best["seg"]
    polished = []
    for k in (0, 1):
        idx = np.where(seg == k)[0]
        old = best["models"][k]
        if len(idx) < 10:
            polished.append(old)
            continue
        new = learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=t_polish)
        better = (predict(new, X[idx]) == y[idx]).mean() >= (predict(old, X[idx]) == y[idx]).mean()
        polished.append(new if better else old)
    new_seg, E = reassign(polished, X, y, seg)
    score = (E.min(axis=1) == 0).mean()
    if score >= best["score"]:
        best.update(score=score, models=polished, seg=new_seg, E=E)

    best["history"] = pd.DataFrame(history)
    best["init"] = init
    return best

def learn_B(X, y, p, inits=INITS, seed=0, n_jobs=None, **kwargs):
    # lance un départ par élément de `inits`, en parallèle (n_jobs=1 : séquentiel), et garde le meilleur
    n_jobs = len(inits) if n_jobs is None else n_jobs
    jobs = (delayed(learn_2_mrsort)(X, y, p, init=how, seed=seed * 100 + r, **kwargs) for r, how in enumerate(inits))
    runs = Parallel(n_jobs=n_jobs, backend="loky")(jobs)
    return max(runs, key=lambda r: r["score"]), runs

### Choix techniques

| Choix | Pourquoi |
|---|---|
| `learn_B` renvoie `(best, runs)` | `best` contient `models`, `seg`, `E`, `score`, `init`, `history`. `runs` garde tous les départs pour le diagnostic. |
| Graines `seed * 100 + r` (comme le jalon 2) | Les départs de `learn_B` et de la version séquentielle du jalon 2 partent des mêmes graines. |
| `learn_mrsort` ne renvoie plus `_lincs` ni `_problem` | Ces objets lincs ne se sérialisent pas entre processus, et T3 et T4 n'utilisent que le numpy. L'accord numpy / lincs de `predict` est testé en T1. |
| `n_jobs=1` désactive le parallélisme | Sert de référence séquentielle pour la validation et le débogage. |
| `fit_segments` sans `shared` | `share_ambiguous` n'est plus géré (voir plus haut). |

### Essai sur un scénario du jalon 3

Un scénario T1 (300 objets de A, 300 du malveillant) passé à `learn_B`. La vérité terrain n'est utilisée que pour **afficher** la pureté de la partition : B ne la voit pas.

In [ ]:
sc = make_scenario(seed=0)
t = time.time()
best, runs = learn_B(sc.X, sc.y, sc.config.p, seed=0)
temps = time.time() - t

sizes = np.bincount(best["seg"], minlength=2)
accord = max((best["seg"] == sc.source).mean(), (best["seg"] != sc.source).mean())     # segments définis à une permutation près
print(f"{temps:.0f} s | départ retenu : {best['init']} | objets expliqués par au moins un modèle : {best['score']:.3f}")
print(f"tailles des segments : {sizes.tolist()} | accord segments / sources : {accord:.3f}")
for r in runs:
    print(f"   départ {r['init']:6s} -> {r['score']:.3f}  ({len(r['history'])} itérations)")

### Validation sur les fichiers du jalon 2

**Question.** Les accélérations (arrêt à 1 %, moins d'itérations, départs en parallèle) dégradent-elles le résultat par rapport à l'algorithme du jalon 2 ?

**Protocole** (celui du jalon 2) : 10 % des objets en test, jamais utilisés pour décider. Mesures : `deux_test` (part des objets de test bien classés par au moins un des deux modèles), `erreur_proportion` (écart entre la taille du petit segment trouvé et la taille attendue, lue dans le nom du fichier), `temps`. Fichiers : `n0600_rep50-50_noise00` de 4 dossiers (le cas 50/50 sans bruit, proche du jalon 3). Mêmes graines pour toutes les variantes.

**Variantes**
- *jalon 2* : séquentiel, `MAX_ITER = 20`, arrêt strict (`TOL_CHANGES = 0`) ;
- *parallèle, 4 départs* avec `TOL_CHANGES = 0,01` et `MAX_ITER` = 8, 10, 20 ;
- *parallèle, 2 départs* (1 guidé + 1 aléatoire).

**Critère de choix** : la configuration la plus rapide dont `deux_test` ne baisse pas de plus de 1 à 2 points par rapport au jalon 2. L'objectif de temps est de moins de 60 s pour 600 objets.

Par défaut la cellule relit `t2_validation.csv`. Passer `RUN_T2 = True` pour relancer les mesures (long : plusieurs dizaines de minutes).

In [ ]:
DATA_DIR = Path("../data/Datasets-Step2")

def load_dataset(csv_path):
    csv_path = Path(csv_path)
    pb = yaml.safe_load((csv_path.parent / "problem.yml").read_text())
    criteria = [str(c["name"]) for c in pb["criteria"]]
    categories = [str(c["name"]) for c in pb["ordered_categories"]]
    df = pd.read_csv(csv_path)
    y = df["category"].astype(str).map({c: h for h, c in enumerate(categories)})
    return df[criteria].to_numpy(float), y.to_numpy(int), len(categories)

def split(N, seed, test_frac=0.10):
    rng = np.random.default_rng(10_000 + seed)
    test = np.zeros(N, bool)
    test[rng.choice(N, round(test_frac * N), replace=False)] = True
    return test

def two_models_accuracy(models, X, y):
    return np.stack([predict(m, X) == y for m in models]).any(axis=0).mean()

VARIANTES = {
    "jalon 2 (séquentiel, 20 it., arrêt strict)": dict(n_jobs=1, max_iter=20, tol_changes=0.0),
    "parallèle, 4 départs, 8 it.":  dict(max_iter=8),
    "parallèle, 4 départs, 10 it.": dict(max_iter=10),
    "parallèle, 4 départs, 20 it.": dict(max_iter=20),
    "parallèle, 2 départs, 10 it.": dict(max_iter=10, inits=("errors", "random")),
}

def bench_t2(files, seeds, variantes, out_csv=None):
    rows = []
    for f in files:
        X, y, p = load_dataset(f)
        rep = re.search(r"rep(\d+)-(\d+)", f.name)
        attendu = min(int(rep[1]), int(rep[2])) / 100
        for s in seeds:
            test = split(len(X), s)
            for nom, kw in variantes.items():
                t = time.time()
                best, _ = learn_B(X[~test], y[~test], p, seed=s, **kw)
                sizes = np.bincount(best["seg"], minlength=2)
                rows.append(dict(dossier=f"{f.parent.parent.name}/{f.parent.name}", fichier=f.stem, graine=s, variante=nom,
                                 deux_train=best["score"], deux_test=two_models_accuracy(best["models"], X[test], y[test]),
                                 erreur_proportion=abs(sizes.min() / sizes.sum() - attendu), depart=best["init"],
                                 iterations=len(best["history"]), temps=time.time() - t))
                if out_csv:
                    pd.DataFrame(rows).to_csv(out_csv, index=False)            # sauvegarde au fil de l'eau
                r = rows[-1]
                print(f"{r['dossier']} graine={s} {nom:45s} test={r['deux_test']:.3f} temps={r['temps']:.0f} s", flush=True)
    return pd.DataFrame(rows)

T2_FILES = [DATA_DIR / d / "n0600_rep50-50_noise00.csv"
            for d in ("type_a/cr06_ca03", "type_a/cr08_ca04", "type_b/cr06_ca03", "type_b/cr08_ca04")]
RUN_T2 = False
t2 = (bench_t2(T2_FILES, seeds=[0], variantes=VARIANTES, out_csv="../results/t2_validation.csv")
      if RUN_T2 else pd.read_csv("../results/t2_validation.csv"))

In [ ]:
ref = list(VARIANTES)[0]
ordre = [v for v in VARIANTES if v in set(t2.variante)]
res = t2.groupby("variante")[["deux_test", "erreur_proportion", "iterations", "temps"]].agg(["mean", "std"]).reindex(ordre)
res.columns = [f"{a} ({b})" for a, b in res.columns]
ecart = (t2.pivot_table(index=["fichier", "dossier", "graine"], columns="variante", values="deux_test")
         .sub(t2.pivot_table(index=["fichier", "dossier", "graine"], columns="variante", values="deux_test")[ref], axis=0))
res["écart deux_test vs jalon 2"] = ecart.mean().reindex(ordre)
display(res.round(3))
display(t2.pivot_table(index=["dossier"], columns="variante", values="deux_test")[ordre].round(3))

### Conclusion de la validation

**On retient 4 départs en parallèle, `MAX_ITER = 10`, `TOL_CHANGES = 0,01`.**

- **Temps** : environ 41 s en moyenne par fichier de 600 objets (29 à 53 s), contre 418 s au jalon 2 (122 à 1081 s). L'objectif de 60 s est tenu, à environ 10 fois plus rapide.
- **Précision** : `deux_test` vaut 0,954 contre 0,967 au jalon 2, soit 1,2 point de moins (écart apparié : écart-type de 3,2 points). C'est la seule variante rapide à rester dans les 1 à 2 points visés.
- **Le nombre de départs ne change pas le temps** en parallèle (2 départs : 39 s, 4 départs : 41 s), mais 2 départs perdent 5,4 points. On garde 4 départs.
- **`MAX_ITER`** : 8 itérations perdent 5 points, et 20 itérations perdent 3,7 points tout en coûtant plus cher (67 s). L'ordre n'est pas monotone : avec 4 fichiers, une graine et 60 objets de test (1,7 point par objet), les écarts de 3 à 5 points sont du même ordre que le bruit. Ces mesures montrent que 10 itérations ne dégradent pas nettement le jalon 2, pas que 10 est strictement meilleur que 8 ou 20.

**Limites à garder en tête**
- `erreur_proportion` se dégrade : 0,071 pour la configuration retenue contre 0,033 au jalon 2. Le petit segment est moins bien dimensionné. Cela pèse sur T3 (identification du modèle de A).
- `deux_train` baisse aussi (0,974 contre 0,996) : moins d'itérations, un ajustement moins poussé.
- lincs est lui-même multithread (OpenMP) : un apprentissage seul utilise environ 5 cœurs. Quatre départs parallèles se partagent donc les cœurs, et chaque apprentissage, limité à 1 s, fait moins de travail (précision d'apprentissage de 0,75 seul contre 0,71 en parallèle). Le gain de temps vient donc surtout des itérations en moins et de l'arrêt à 1 %.
- Un essai de la variante à 8 itérations a pris 486 s sur un fichier (contre 46 à 50 s ailleurs), sans cause identifiée : à surveiller.

## T3. Identifier le modèle de A

Après T2, B a **deux modèles** et un rangement de chaque objet dans l'un des deux segments, mais les segments « 0 » et « 1 » n'ont pas de sens : B ne sait pas lequel est celui de A.

**Un problème d'identifiabilité.** Avec les choix de T1 (même générateur de modèles, même loi des $x$) et un ratio de 1, les deux sources sont **parfaitement symétriques** : choisir A serait un pile ou face, même avec une séparation parfaite. L'information n'est pas dans les données. Il faut une asymétrie. On exploite celle de l'énoncé : le malveillant « peut » envoyer une donnée additionnelle pour chaque donnée de A, donc il envoie **au plus autant** que A. S'il envoie moins, **A est majoritaire** dans ce que reçoit B.

| Choix | Décision |
|---|---|
| Asymétrie exploitée | ratio du malveillant < 1 : A est majoritaire |
| Ratio par défaut | 0,5 (A représente 67 % de ce que reçoit B). Balayage en T4 sur {1,0 ; 0,85 ; 0,7 ; 0,5} |
| Indicateur de confiance | **objets expliqués** : sur le mélange entier, chaque modèle appris explique $n_k$ objets (il les classe dans la catégorie reçue) ; confiance $= |n_0 - n_1| / N$ |
| Décision | confiance $\ge$ seuil : B choisit le modèle qui explique le plus d'objets ; sinon **abstention** : B renvoie les deux modèles |
| Seuil | `SEUIL_CONFIANCE` : provisoire (0 : pas d'abstention) jusqu'à la calibration de T4, qui fixe `SEUIL_CALIBRE` |
| Évaluation | deux métriques : **oracle** (qualité de la séparation) et **identifié** (qualité du choix) |
| Méthodes de comparaison | `identify_A_taille` (M2, choix par la taille du segment) et `learn_single` (M1, un seul modèle) |

*Pourquoi les objets expliqués plutôt que la taille des segments.* La première version choisissait le plus grand segment. Les essais ont montré que `learn_B` **équilibre** les segments (il trouve 261/189 pour 300/150 réels, et même 365/235 pour 300/300) : l'écart de taille est déformé par l'algorithme. Le nombre d'objets expliqués ne dépend que des deux modèles et des données reçues, pas du rangement. L'asymétrie exploitée est la même (A est majoritaire) ; elle se lit seulement sur les données. Le diagnostic (section suivante) a donné 18 bons choix sur 18 à ratio < 1 pour ce critère, contre 14 sur 18 pour la taille.

*Pourquoi pas la qualité d'ajustement du modèle :* rien ne rend le modèle de A mieux ajusté que celui du malveillant, cela créerait un signal artificiel. Le critère d'objets expliqués compte les objets sur les **deux** sources et n'avantage A que par son effectif.

*Pourquoi l'abstention :* quand les comptes sont proches, B annoncerait une certitude qu'il n'a pas. Le cas ratio = 1 est le pire cas pour B et le meilleur pour le malveillant.

In [ ]:
from dataclasses import dataclass

SEUIL_CONFIANCE = 0.0       # PROVISOIRE : pas d'abstention tant que T4 n'a pas calibré le seuil (SEUIL_CALIBRE). Écart de comptes, en part de N.
SEUIL_TAILLE    = 0.10      # seuil a priori de l'ancien critère « taille » (identify_A_taille), conservé pour la comparaison M2

@dataclass
class Decision:
    abstention: bool                  # True : B ne choisit pas et renvoie les deux modèles
    confiance: float                  # objets expliqués : |n_0 - n_1| / N ; taille : (n_grand - n_petit) / N ; entre 0 et 1
    indice_segment_A: int | None      # segment supposé être celui de A (None si abstention)
    modele_A: dict | None             # modèle supposé être celui de A (None si abstention)
    modeles: list | None              # les deux modèles (renseigné seulement en cas d'abstention)
    tailles: tuple                    # (taille du segment 0, taille du segment 1)
    seuil: float
    expliques: tuple | None = None    # (n_0, n_1) : objets du mélange expliqués par chaque modèle (None pour le critère « taille »)

def explained_counts(result_B, X, y):
    # nombre d'objets du mélange (X, y) que chaque modèle classe dans la catégorie reçue
    return np.array([int((predict(m, X) == y).sum()) for m in result_B["models"]])

def identify_A(result_B, X, y, seuil=None):
    # Critère « objets expliqués » (méthode de B). result_B : meilleur résultat de learn_B (clés `seg` et `models`) ;
    # (X, y) : le mélange reçu par B. Le modèle qui explique le plus d'objets est supposé être celui de A (A est
    # majoritaire quand le ratio du malveillant est < 1) ; en cas d'égalité, le segment 0. Ne dépend pas de la partition.
    seuil = SEUIL_CONFIANCE if seuil is None else seuil
    models = result_B["models"]
    n = explained_counts(result_B, X, y)
    tailles = np.bincount(result_B["seg"], minlength=2)
    confiance = abs(int(n[0]) - int(n[1])) / len(y)
    t, e = (int(tailles[0]), int(tailles[1])), (int(n[0]), int(n[1]))
    if confiance < seuil:
        return Decision(True, confiance, None, None, list(models), t, seuil, e)
    k = int(np.argmax(n))
    return Decision(False, confiance, k, models[k], None, t, seuil, e)

def identify_A_taille(result_B, seuil=SEUIL_TAILLE):
    # Ancien critère (M2) : le plus grand segment est supposé être celui de A ; confiance = (n_grand - n_petit) / N.
    # Conservé pour la comparaison : la partition de learn_B équilibre les segments, ce qui le rend peu fiable.
    seg, models = result_B["seg"], result_B["models"]
    tailles = np.bincount(seg, minlength=2)
    confiance = abs(int(tailles[0]) - int(tailles[1])) / len(seg)
    t = (int(tailles[0]), int(tailles[1]))
    if confiance < seuil:
        return Decision(True, confiance, None, None, list(models), t, seuil)
    k = int(np.argmax(tailles))
    return Decision(False, confiance, k, models[k], None, t, seuil)

def run_B(X, y, p, seed=0, seuil=None):
    # méthode de B de bout en bout (M3) : learn_B (T2) puis identify_A (T3, objets expliqués)
    best, _ = learn_B(X, y, p, seed=seed)
    return best, identify_A(best, X, y, seuil)

### Évaluation : séparation et identification

Deux métriques, mesurées sur le jeu de test du scénario (objets uniformes, **étiquetés par le modèle de A**) :

| Mesure | Définition | Ce qu'elle dit |
|---|---|---|
| `oracle` | précision du meilleur des deux modèles appris | qualité de la **séparation** (T2) : B a-t-il appris le modèle de A, quel que soit le choix ? |
| `identifie` | précision du modèle choisi par B ; **NaN en cas d'abstention** | qualité du **choix** (T3) : B a-t-il rendu le bon des deux ? |
| `bon_choix` | le modèle choisi est celui des deux dont la précision en test est la plus haute ; NaN en cas d'abstention | définition a posteriori du « bon choix » (utilisée par le diagnostic et T4) |
| `bon_segment` | le segment choisi contient majoritairement des objets de A (d'après `source`) ; NaN en cas d'abstention | l'identification est correcte, au sens de la partition |
| `grand_segment_est_A` | même test pour le plus grand segment, **abstention ou non** | l'hypothèse « le plus grand segment est A » est-elle juste ? (critère « taille ») |
| `accord` | part des objets bien rangés, à une permutation près | pureté de la partition |

`identifie` vaut NaN en cas d'abstention : l'abstention est comptée à part (`abstention`, taux en T4), et la précision moyenne des cas où B se prononce se calcule avec `nanmean`. On ne remplace pas NaN par l'`oracle`, ce qui masquerait le coût de l'abstention.

`identifie` $\le$ `oracle` toujours. Si `oracle` est bon et `identifie` mauvais, l'échec vient du choix (T3) ; si `oracle` est mauvais, il vient de la séparation (T2).

In [ ]:
def model_accuracy(model, X, y):
    return float((predict(model, X) == y).mean())

def evaluate_identification(scenario, result_B, decision):
    # Compare le résultat de B à la vérité terrain du scénario (que B ne voit pas).
    seg, models, source = result_B["seg"], result_B["models"], scenario.source
    acc = [model_accuracy(m, scenario.X_test, scenario.y_test) for m in models]
    part_A = [float((source[seg == k] == 0).mean()) if (seg == k).any() else np.nan for k in (0, 1)]   # part d'objets de A par segment
    grand = int(np.argmax(np.bincount(seg, minlength=2)))
    choisi = decision.indice_segment_A
    return dict(
        abstention=bool(decision.abstention), confiance=float(decision.confiance),
        oracle=max(acc), oracle_indice=int(np.argmax(acc)),
        identifie=np.nan if decision.abstention else acc[choisi],
        bon_choix=np.nan if decision.abstention else float(acc[choisi] >= max(acc)),
        bon_segment=np.nan if decision.abstention else float(part_A[choisi] > 0.5),
        grand_segment_est_A=float(part_A[grand] > 0.5),
        accord=float(max((seg == source).mean(), (seg != source).mean())),
        taille_grand=int(max(decision.tailles)), taille_petit=int(min(decision.tailles)),
        n_A=int((source == 0).sum()), n_M=int((source == 1).sum()))

### Tests de T3 (sans lincs)

Résultats de B **synthétiques** : on construit un `result_B` à partir de la vérité terrain d'un scénario T1 (partition parfaite ou volontairement fausse, modèles vrais), ce qui rend les tests déterministes et instantanés.

**Critère « objets expliqués » (`identify_A`)**
1. ratio 0,5, partition parfaite : B choisit le modèle de A, qui explique au moins 300 objets contre au moins 150, `oracle` = `identifie` = 1 ;
2. mêmes données avec les numéros de segments **permutés** : même identification (les numéros n'ont pas de sens) ;
3. deux modèles identiques : comptes égaux, confiance 0, abstention dès que le seuil est strictement positif ; seuil 0 : B choisit le segment 0 ;
4. seuil au-dessus de toute confiance : abstention, `identifie` = NaN, `oracle` = 1 ;
5. frontière du seuil : une confiance exactement égale au seuil donne un choix, juste au-dessus une abstention ;
6. limite : si le malveillant envoie **plus** que A (ratio 2), l'hypothèse est fausse et B choisit le modèle du malveillant, avec `oracle` = 1 ;
7. le choix **ne dépend pas de la partition** : avec un rangement aléatoire des objets (`accord` proche de 0,5), B choisit encore le modèle de A.

**Ancien critère « taille » (`identify_A_taille`)**
8. partition parfaite, ratio 0,5 : choix du segment de A (confiance 1/3) ; numéros permutés ; ratio 1 : tailles égales, abstention au seuil de 0,10 ; seuil 0 : choix du segment 0 ; frontière du seuil.

In [ ]:
def perfect_result(sc, swap=False):
    # résultat de B si la séparation était parfaite ; swap : numéros de segments permutés
    seg = (1 - sc.source) if swap else sc.source.copy()
    models = [sc.model_M, sc.model_A] if swap else [sc.model_A, sc.model_M]
    return dict(seg=seg, models=models)

sc5 = make_scenario(ScenarioConfig(ratio_malveillant=0.5), seed=11)         # 300 contre 150

# 1. séparation parfaite : le choix est le bon
r = perfect_result(sc5); d = identify_A(r, sc5.X, sc5.y, seuil=0.0); e = evaluate_identification(sc5, r, d)
n = explained_counts(r, sc5.X, sc5.y)
assert n[0] >= 300 and n[1] >= 150 and n[0] > n[1]
assert not d.abstention and d.indice_segment_A == 0 and d.modele_A is sc5.model_A and d.modeles is None
assert d.expliques == (int(n[0]), int(n[1])) and np.isclose(d.confiance, (n[0] - n[1]) / len(sc5.y)) and d.tailles == (300, 150)
assert e["oracle"] == e["identifie"] == 1.0 and e["bon_choix"] == 1.0 and e["bon_segment"] == 1.0 and e["accord"] == 1.0

# 2. numéros de segments permutés : même identification
r = perfect_result(sc5, swap=True); d = identify_A(r, sc5.X, sc5.y, seuil=0.0); e = evaluate_identification(sc5, r, d)
assert d.indice_segment_A == 1 and d.modele_A is sc5.model_A and e["identifie"] == 1.0 and e["bon_segment"] == 1.0

# 3. deux modèles identiques : comptes égaux, confiance 0
r_id = dict(seg=sc5.source.copy(), models=[sc5.model_A, sc5.model_A])
d = identify_A(r_id, sc5.X, sc5.y, seuil=0.01)
assert d.abstention and d.confiance == 0 and d.modele_A is None and d.indice_segment_A is None and len(d.modeles) == 2
d = identify_A(r_id, sc5.X, sc5.y, seuil=0.0)
assert not d.abstention and d.indice_segment_A == 0

# 4. seuil au-dessus de toute confiance : abstention
r = perfect_result(sc5)
d = identify_A(r, sc5.X, sc5.y, seuil=1.0); e = evaluate_identification(sc5, r, d)
assert d.abstention and np.isnan(e["identifie"]) and np.isnan(e["bon_choix"]) and np.isnan(e["bon_segment"]) and e["oracle"] == 1.0

# 5. frontière : confiance == seuil -> choix ; juste au-dessus -> abstention
c = identify_A(r, sc5.X, sc5.y, seuil=0.0).confiance
assert not identify_A(r, sc5.X, sc5.y, seuil=c).abstention and identify_A(r, sc5.X, sc5.y, seuil=c + 1e-9).abstention

# 6. limite : si le malveillant envoie plus que A, B choisit le modèle du malveillant
sc2 = make_scenario(ScenarioConfig(ratio_malveillant=2.0), seed=11)
r = perfect_result(sc2); d = identify_A(r, sc2.X, sc2.y, seuil=0.0); e = evaluate_identification(sc2, r, d)
assert not d.abstention and d.modele_A is sc2.model_M
assert e["bon_choix"] == 0.0 and e["bon_segment"] == 0.0 and e["oracle"] == 1.0 and e["identifie"] < e["oracle"]

# 7. le choix ne dépend pas de la partition
rng = np.random.default_rng(0)
r = dict(seg=rng.integers(0, 2, len(sc5.X)), models=[sc5.model_A, sc5.model_M])
d = identify_A(r, sc5.X, sc5.y, seuil=0.0); e = evaluate_identification(sc5, r, d)
assert 0.45 < e["accord"] < 0.62 and d.indice_segment_A == 0 and e["bon_choix"] == 1.0

# 8. ancien critère « taille »
sc1 = make_scenario(ScenarioConfig(ratio_malveillant=1.0), seed=11)
r = perfect_result(sc5); d = identify_A_taille(r)
assert not d.abstention and d.indice_segment_A == 0 and d.modele_A is sc5.model_A and np.isclose(d.confiance, 1 / 3) and d.expliques is None
d = identify_A_taille(perfect_result(sc5, swap=True)); assert d.indice_segment_A == 1 and d.modele_A is sc5.model_A
d = identify_A_taille(perfect_result(sc1)); assert d.abstention and d.confiance == 0
assert not identify_A_taille(perfect_result(sc1), seuil=0.0).abstention and identify_A_taille(perfect_result(sc1), seuil=0.0).indice_segment_A == 0
assert not identify_A_taille(r, seuil=1 / 3).abstention and identify_A_taille(r, seuil=1 / 3 + 1e-9).abstention
print("tests T3 : ok (8 groupes)")

### Essai réel sur des scénarios T1

Chaîne complète : `make_scenario` → `learn_B` → `identify_A_taille` → `evaluate_identification`, sur quelques scénarios (ce n'est pas le benchmark de T4). Les graines 1 à 3 en ratio 0,5, et la graine 1 en ratio 1,0 (même modèles de A et du malveillant qu'en ratio 0,5) pour montrer l'abstention attendue.

**Cet essai a été fait avec le critère « taille » et son seuil a priori de 0,10** (la décision de départ). C'est lui qui a montré que la taille des segments est un mauvais indicateur, d'où le diagnostic et le passage au critère « objets expliqués ». Il est conservé tel quel pour l'historique.

Chaque scénario coûte environ 30 à 40 s. Par défaut la cellule relit `t3_essais.csv`. Passer `RUN_T3 = True` pour relancer.

In [ ]:
ESSAIS_T3 = [(1, 0.5), (2, 0.5), (3, 0.5), (1, 1.0)]       # (graine, ratio du malveillant)

def essai_t3(seed, ratio, seuil=SEUIL_TAILLE):
    sc = make_scenario(ScenarioConfig(ratio_malveillant=ratio), seed)
    t = time.time()
    best, _ = learn_B(sc.X, sc.y, sc.config.p, seed=seed)
    temps = time.time() - t
    ev = evaluate_identification(sc, best, identify_A_taille(best, seuil))
    return dict(graine=seed, ratio=ratio, **ev, temps=temps)

RUN_T3 = False
if RUN_T3:
    rows = []
    for s, ratio in ESSAIS_T3:
        rows.append(essai_t3(s, ratio))
        pd.DataFrame(rows).to_csv("../results/t3_essais.csv", index=False)            # sauvegarde au fil de l'eau
        print(rows[-1], flush=True)
t3 = pd.read_csv("../results/t3_essais.csv")
cols = ["graine", "ratio", "n_A", "n_M", "taille_grand", "taille_petit", "confiance", "abstention",
        "oracle", "identifie", "bon_segment", "grand_segment_est_A", "accord", "temps"]
display(t3[cols].round(3))

### Résultats de l'essai réel et limites (critère « taille »)

| Constat | Chiffres |
|---|---|
| Ratio 0,5 : l'identification est correcte sur les 3 scénarios | `bon_segment` = 1 ; `identifie` = `oracle` = 0,88 / 0,86 / 0,88 |
| Les segments trouvés sont **plus équilibrés que les vrais** | trouvés 285/165, 261/189, 305/145 contre 300/150 réels ; confiance 0,27 / 0,16 / 0,36 contre 0,33 réelle |
| Le scénario d'exemple (graine 0) donnerait une abstention | segments 230/220, confiance 0,02 (accord 0,73) |
| **Ratio 1,0 : B ne s'abstient pas et se trompe** | segments trouvés 365/235 pour 300/300 réels, confiance 0,22 > seuil de 0,10 ; `bon_segment` = 0, `identifie` = 0,44 pour un `oracle` de 0,88 |
| La séparation reste moyenne | `oracle` ≈ 0,88 : même le meilleur des deux modèles classe 12 % du test autrement que le modèle de A |

**À retenir.** Ces résultats ont conduit au diagnostic puis au critère « objets expliqués » (section suivante), qui est maintenant celui de `identify_A`. Pour la suite de la lecture : À ratio 1, les tailles trouvées s'écartent de 50/50 de plusieurs dizaines d'objets sans aucun signal : une confiance de 0,22 ne prouve rien, et le seuil provisoire de 0,10 est trop bas. L'indicateur de confiance doit être calibré sur des scénarios à ratio 1, où l'on connaît la bonne réponse (aucune), pour fixer un seuil qui limite les faux choix. Quatre essais ne suffisent pas à le faire : c'est le rôle de T4. Détails dans `docs/jalon3.md`.

## Diagnostic T3 : le pipeline à deux modèles apporte-t-il quelque chose ?

Les premiers essais de T3 posent deux questions que la taille des segments ne permet pas de trancher :

1. **Faut-il vraiment séparer les deux sources ?** Si A est majoritaire, un **seul** modèle appris sur tout le mélange devrait déjà s'ajuster à A : les objets du malveillant sont alors du bruit pour ce modèle. C'est la méthode la plus simple.
2. **La taille d'un segment est-elle un bon indicateur ?** `learn_B` tend à équilibrer les segments, donc l'écart de taille qu'il trouve est déformé. Un autre critère n'utilise pas la partition : le nombre d'objets du mélange entier que chaque modèle appris **explique**, c'est-à-dire qu'il classe dans la catégorie reçue. L'asymétrie exploitée est la même (A est majoritaire), mais elle se lit sur les données et non sur le rangement fait par l'algorithme.

**Méthodes comparées** (même scénario, mêmes données, toutes évaluées sur le jeu de test étiqueté par le modèle de A) :

| | Méthode |
|---|---|
| **M1** | un seul modèle lincs sur tout le mélange (4 apprentissages en parallèle, on garde le meilleur sur le mélange reçu) |
| **M2** | `learn_B` (2 modèles) puis choix par **taille de segment**, sans abstention |
| **M3** | `learn_B` (2 modèles) puis choix par **objets expliqués**, sans abstention |
| oracle | le meilleur des deux modèles de `learn_B` (qualité de la séparation seule) |
| hasard | la moyenne des deux modèles de `learn_B` (choisir au pif) |

**Scénarios.** Ratios {1,0 ; 0,85 ; 0,7 ; 0,5} et 6 graines par ratio (graines 100 à 105). Ce sont des scénarios de **diagnostic**, distincts de ceux de l'essai T3 (graines 1 à 3) ; ils ne serviront pas au rapport final. Aucune méthode n'utilise la vérité terrain pour décider : B ne voit que `X` et `y`.

Le « bon choix » est défini a posteriori : le modèle choisi est celui des deux dont la précision en test est la plus haute. À ratio 1,0 les deux sources sont symétriques, donc la notion y est peu significative (on la rapporte quand même).

In [ ]:
def _fit_single(X, y, p, seed, seconds):
    m = learn_mrsort(X, y, p, seed=seed, seconds=seconds)
    return m, float((predict(m, X) == y).mean())

def learn_single(X, y, p, seed=0, n_starts=4, polish_seconds=POLISH_SECONDS, n_jobs=None):
    # M1, MÉTHODE DE RÉFÉRENCE : un seul modèle MR-Sort sur tout le mélange (T4 compare B à elle). n_starts apprentissages (graines différentes) en parallèle,
    # comme les départs de learn_B ; on garde celui qui a la meilleure précision SUR LE MÉLANGE REÇU (pas de vérité terrain).
    t = time.time()
    seconds = budget(polish_seconds, X.shape[1], p)
    jobs = (delayed(_fit_single)(X, y, p, seed * 100 + r, seconds) for r in range(n_starts))
    runs = Parallel(n_jobs=n_starts if n_jobs is None else n_jobs, backend="loky")(jobs)
    model, acc = max(runs, key=lambda r: r[1])
    return dict(model=model, train_accuracy=acc, accuracies=[a for _, a in runs], temps=time.time() - t)

### Tests rapides de `learn_single`

lincs s'arrête au bout d'un temps (`TerminateAfterSeconds`), donc deux appels avec la même graine ne donnent pas exactement le même modèle : on vérifie seulement que la précision d'apprentissage est stable à quelques points près. Le critère « objets expliqués » est testé avec ceux de T3.

In [ ]:
sc_s = make_scenario(ScenarioConfig(n_A=100, ratio_malveillant=0.5), seed=21)

# learn_single : forme du modèle, précision d'apprentissage dans [0, 1], meilleure des 4, reproductibilité approchée
r1 = learn_single(sc_s.X, sc_s.y, 3, seed=7, polish_seconds=0.5)
m = r1["model"]
assert m["profiles"].shape == (2, 6) and m["raw_weights"].shape == (6,) and np.isclose(m["weights"].sum(), 1)
assert len(r1["accuracies"]) == 4 and all(0 <= a <= 1 for a in r1["accuracies"])
assert r1["train_accuracy"] == max(r1["accuracies"])
assert np.isclose(r1["train_accuracy"], (predict(m, sc_s.X) == sc_s.y).mean())
r2 = learn_single(sc_s.X, sc_s.y, 3, seed=7, polish_seconds=0.5)
assert abs(r1["train_accuracy"] - r2["train_accuracy"]) < 0.06
learn_single(sc_s.X, sc_s.y, 3, seed=8, polish_seconds=0.5)                 # une autre graine ne plante pas
print("tests learn_single : ok")

### Expérience de diagnostic

`RUN_DIAG = True` relance les 24 scénarios (environ 40 s chacun, soit 15 à 20 minutes) ; l'expérience reprend où elle s'était arrêtée si `results/t3_diagnostic.csv` existe déjà. Par défaut, la cellule relit le CSV.

In [ ]:
RATIOS_DIAG  = (1.0, 0.85, 0.7, 0.5)
GRAINES_DIAG = range(100, 106)         # scénarios de diagnostic : distincts de ceux de l'essai T3 (graines 1 à 3)
DIAG_CSV     = "../results/t3_diagnostic.csv"

def diag_scenario(seed, ratio):
    sc = make_scenario(ScenarioConfig(ratio_malveillant=ratio), seed)
    p = sc.config.p
    acc_test = lambda mod: model_accuracy(mod, sc.X_test, sc.y_test)

    t = time.time()
    single = learn_single(sc.X, sc.y, p, seed=seed)
    t_m1 = time.time() - t

    t = time.time()
    best, _ = learn_B(sc.X, sc.y, p, seed=seed)
    t_pipe = time.time() - t

    d2 = identify_A_taille(best, seuil=0.0)                            # M2 : plus grand segment
    d3 = identify_A(best, sc.X, sc.y, seuil=0.0)                       # M3 : objets expliqués
    acc = [acc_test(mod) for mod in best["models"]]
    n_expl = explained_counts(best, sc.X, sc.y)
    part_A = [float((sc.source[best["seg"] == k] == 0).mean()) if (best["seg"] == k).any() else np.nan for k in (0, 1)]
    return dict(
        graine=seed, ratio=ratio, n_A=int((sc.source == 0).sum()), n_M=int((sc.source == 1).sum()),
        taille_0=int(d2.tailles[0]), taille_1=int(d2.tailles[1]),
        taille_grand=int(max(d2.tailles)), taille_petit=int(min(d2.tailles)),
        accord=float(max((best["seg"] == sc.source).mean(), (best["seg"] != sc.source).mean())),
        m1=acc_test(single["model"]), m1_train=single["train_accuracy"],
        m2=acc[d2.indice_segment_A], m3=acc[d3.indice_segment_A],
        oracle=max(acc), hasard=float(np.mean(acc)), acc_0=acc[0], acc_1=acc[1],
        choix_m2=d2.indice_segment_A, choix_m3=d3.indice_segment_A, meilleur=int(np.argmax(acc)),
        bon_m2=float(acc[d2.indice_segment_A] >= max(acc)), bon_m3=float(acc[d3.indice_segment_A] >= max(acc)),
        part_A_0=part_A[0], part_A_1=part_A[1],
        conf_m2=d2.confiance, conf_m3=d3.confiance, expl_0=int(n_expl[0]), expl_1=int(n_expl[1]),
        temps_m1=t_m1, temps_pipeline=t_pipe)

RUN_DIAG = False
if RUN_DIAG:
    try:
        rows = pd.read_csv(DIAG_CSV).to_dict("records")
    except FileNotFoundError:
        rows = []
    deja = {(int(r["graine"]), float(r["ratio"])) for r in rows}
    for s in GRAINES_DIAG:
        for ratio in RATIOS_DIAG:
            if (s, ratio) in deja:
                continue
            rows.append(diag_scenario(s, ratio))
            pd.DataFrame(rows).to_csv(DIAG_CSV, index=False)                  # sauvegarde au fil de l'eau
            r = rows[-1]
            print(f"graine={s} ratio={ratio}: M1={r['m1']:.3f} M2={r['m2']:.3f} M3={r['m3']:.3f} "
                  f"oracle={r['oracle']:.3f} | {r['temps_pipeline']:.0f} s", flush=True)
diag = pd.read_csv(DIAG_CSV)
print(f"{len(diag)} scénarios de diagnostic : ratios {sorted(diag.ratio.unique(), reverse=True)}, "
      f"{diag.groupby('ratio').size().min()} à {diag.groupby('ratio').size().max()} graines par ratio")

### Résultats du diagnostic

**Attention : n = 6 scénarios par ratio.** Les moyennes et écarts-types ci-dessous sont des indices, pas des estimations précises : un écart d'un point de précision ne signifie rien, et un jeu de test de 1000 objets ne limite pas cette incertitude, c'est le nombre de scénarios qui la fixe.

In [ ]:
ORDRE_RATIOS = sorted(diag.ratio.unique(), reverse=True)

def moy_std(serie):
    return f"{serie.mean():.3f} ± {serie.std():.3f}"

# 1) précision sur le jeu de test (modèle de A), par ratio
NOMS = {"m1": "M1 modèle unique", "m2": "M2 pipeline + taille", "m3": "M3 pipeline + expliqués",
        "oracle": "oracle (meilleur des 2)", "hasard": "hasard (moyenne des 2)"}
prec = pd.DataFrame({NOMS[c]: diag.groupby("ratio")[c].apply(moy_std) for c in NOMS}).loc[ORDRE_RATIOS]
print("Précision en test, moyenne ± écart-type sur les graines")
display(prec)

# 2) différences appariées (même scénario)
diag = diag.assign(d_m1_m2=diag.m1 - diag.m2, d_m3_m2=diag.m3 - diag.m2, d_m1_oracle=diag.m1 - diag.oracle)
dif = diag.groupby("ratio").agg(
    **{"M1 − M2": ("d_m1_m2", lambda s: moy_std(s)), "M1 > M2 (sur 6)": ("d_m1_m2", lambda s: int((s > 0).sum())),
       "M3 − M2": ("d_m3_m2", lambda s: moy_std(s)), "M3 > M2 (sur 6)": ("d_m3_m2", lambda s: int((s > 0).sum())),
       "M1 − oracle": ("d_m1_oracle", lambda s: moy_std(s))}).loc[ORDRE_RATIOS]
print("Différences appariées de précision (moyenne ± écart-type) et nombre de scénarios où la première méthode gagne")
display(dif)

In [ ]:
# 3) taux de bon choix, tailles trouvées contre réelles, temps
choix = diag.groupby("ratio").agg(**{"bon choix M2": ("bon_m2", "mean"), "bon choix M3": ("bon_m3", "mean")}).loc[ORDRE_RATIOS]
choix["M2 = M3"] = (diag.choix_m2 == diag.choix_m3).groupby(diag.ratio).mean().loc[ORDRE_RATIOS]
choix["segment le plus grand = A"] = diag.assign(g=(np.where(diag.taille_0 >= diag.taille_1, diag.part_A_0, diag.part_A_1) > 0.5)
                                                 ).groupby("ratio").g.mean().loc[ORDRE_RATIOS]
choix["accord segments / sources"] = diag.groupby("ratio").accord.mean().loc[ORDRE_RATIOS]
choix["taille trouvée (grand / petit)"] = [f"{diag[diag.ratio == r].taille_grand.mean():.0f} / {diag[diag.ratio == r].taille_petit.mean():.0f}"
                                           for r in ORDRE_RATIOS]
choix["taille réelle (A / malveillant)"] = [f"{diag[diag.ratio == r].n_A.mean():.0f} / {diag[diag.ratio == r].n_M.mean():.0f}" for r in ORDRE_RATIOS]
choix["temps M1 (s)"] = diag.groupby("ratio").temps_m1.mean().loc[ORDRE_RATIOS]
choix["temps pipeline (s)"] = diag.groupby("ratio").temps_pipeline.mean().loc[ORDRE_RATIOS]
print("Choix du modèle de A, partitions et temps")
display(choix.round(3))

# 4) la confiance sépare-t-elle les bons des mauvais choix ? (ratios < 1 : à 1,0 il n'y a pas de bonne réponse)
def auc(conf, bon):
    # P(confiance d'un bon choix > confiance d'un mauvais choix), égalités comptées 1/2 ; None s'il manque une classe
    g, b = conf[bon == 1].to_numpy(), conf[bon == 0].to_numpy()
    if len(g) == 0 or len(b) == 0:
        return np.nan
    return float(np.mean([(x > z) + 0.5 * (x == z) for x in g for z in b]))

sous = diag[diag.ratio < 1]
sep = pd.DataFrame({
    "critère": ["taille (M2)", "objets expliqués (M3)"],
    "n choix corrects": [int(sous.bon_m2.sum()), int(sous.bon_m3.sum())],
    "n choix faux": [int((1 - sous.bon_m2).sum()), int((1 - sous.bon_m3).sum())],
    "confiance moyenne, bons choix": [sous[sous.bon_m2 == 1].conf_m2.mean(), sous[sous.bon_m3 == 1].conf_m3.mean()],
    "confiance moyenne, mauvais choix": [sous[sous.bon_m2 == 0].conf_m2.mean(), sous[sous.bon_m3 == 0].conf_m3.mean()],
    "AUC": [auc(sous.conf_m2, sous.bon_m2), auc(sous.conf_m3, sous.bon_m3)]}).set_index("critère")
print("Séparation bons / mauvais choix par la confiance, ratios 0,85 / 0,7 / 0,5 réunis (AUC : 0,5 = aucune information, 1 = parfait)")
display(sep.round(3))
print("Confiance moyenne par ratio")
display(diag.groupby("ratio")[["conf_m2", "conf_m3"]].mean().loc[ORDRE_RATIOS].round(3))

### Lecture des résultats

*Les valeurs ci-dessous sont celles de `results/t3_diagnostic.csv` (24 scénarios, 6 par ratio). Elles sont à lire avec n = 6 par ratio.*

| Ratio | M1 modèle unique | M2 pipeline + taille | M3 pipeline + expliqués | oracle | hasard |
|---|---|---|---|---|---|
| 1,0 | 0,632 ± 0,065 | 0,516 ± 0,275 | 0,724 ± 0,261 | 0,880 ± 0,052 | 0,605 ± 0,059 |
| 0,85 | 0,740 ± 0,094 | 0,671 ± 0,227 | 0,853 ± 0,111 | 0,853 ± 0,111 | 0,600 ± 0,073 |
| 0,7 | 0,798 ± 0,100 | 0,803 ± 0,189 | 0,847 ± 0,105 | 0,847 ± 0,105 | 0,605 ± 0,046 |
| 0,5 | 0,878 ± 0,050 | 0,826 ± 0,175 | 0,893 ± 0,046 | 0,893 ± 0,046 | 0,653 ± 0,070 |

**Ce qui est établi**
- **M3 ne se trompe jamais quand A est majoritaire** : 18 bons choix sur 18 aux ratios 0,85, 0,7 et 0,5 (borne basse à 95 % du taux de bon choix : environ 82 %, Wilson). M2, par la taille, se trompe 4 fois sur 18 (14 bons choix). Les 4 erreurs de M2 viennent de partitions mal rangées (accord 0,58 à 0,75) où le plus grand segment trouvé n'est pas celui de A ; M3 ne passe pas par la partition et choisit juste dans ces 4 cas.
- **La séparation en deux modèles fonctionne même à ratio 1,0** : `oracle` = 0,88, c'est-à-dire que l'un des deux modèles appris approche bien celui de A. Ce qui est impossible à ratio 1,0, c'est de dire lequel (M3 : 4 bons choix sur 6, ce que le hasard donne avec une probabilité de 0,34).
- **Temps** : M1 prend 4,2 s, le pipeline 27 s (environ 6 fois plus).

**Ce qui est indicatif (n = 6)**
- M3 fait mieux que M1 en moyenne à chaque ratio (différence M1 − M3 : −1,4 ; −4,8 ; −11,3 ; −9,2 points pour 0,5 ; 0,7 ; 0,85 ; 1,0), mais M1 gagne dans 1 à 3 scénarios sur 6 selon le ratio, et à ratio 0,5 l'écart (−1,4 ± 2,9) ne se distingue pas de zéro.
- M2 n'est pas meilleur que M1 en moyenne : M1 − M2 va de −0,4 à +11,5 points selon le ratio, avec des écarts-types de 16 à 28 points.
- La taille trouvée des segments (grand / petit, contre réel) : 276/174 (300/150), 301/209 (300/210), 346/210 (300/255), 358/242 (300/300). Le plus grand segment est celui de A dans 6/6, 6/6, 4/6 et 2/6 des scénarios pour 0,5 ; 0,7 ; 0,85 ; 1,0.

**Ce qui n'est pas conclu**
- Le seuil de confiance : M3 n'a commis aucune erreur à ratio < 1, donc aucune confiance de mauvais choix n'est disponible pour fixer un seuil. La confiance des bons choix de M3 va de 0,006 à 0,465 (9 sur 18 sous 0,10, 3 sur 18 sous 0,05). L'AUC de M2 (0,70, 14 bons contre 4 mauvais) est trop fragile pour être interprétée.
- Le comportement à ratio 1,0 : M3 y fait 4/6, sans écart avec le hasard.
- Toute généralisation à d'autres tailles (n, p, n_A) : une seule configuration (6 critères, 3 catégories, 300 objets de A) est testée.

## T4. Évaluation de B contre la vérité terrain

**Question.** Sur beaucoup de scénarios, dans quelle mesure B retrouve-t-il le modèle de A, et à partir de quel ratio du malveillant cela cesse-t-il d'être possible ?

**Protocole décidé** (aucun autre choix n'est introduit) :

| Élément | Décision |
|---|---|
| Ratios du malveillant | {1,0 ; 0,85 ; 0,7 ; 0,5} |
| Configuration | par défaut : 6 critères, 3 catégories, 300 objets de A, loi uniforme. **Aucune autre variation.** |
| Lot de calibration | 10 scénarios par ratio (40), graines **200 à 239** : `200 + 10·i_ratio + k` |
| Lot final | 20 scénarios par ratio (80), graines **300 à 379** : `300 + 20·i_ratio + k` |
| Graines déjà utilisées | 0 à 9 développement, 100 à 105 diagnostic : jamais réutilisées |
| Méthodes (mêmes scénarios) | **M1** modèle unique (`learn_single`, référence) · **M2** `learn_B` + `identify_A_taille` · **M3** `learn_B` + `identify_A` (méthode de B) · références **oracle** (meilleur des deux modèles appris) et **hasard** (moyenne des deux) |
| Mesure | précision sur le jeu de test de 1000 objets étiqueté par le modèle de A |
| Seuil d'abstention | calibré sur le lot de calibration **seulement**, puis figé (`SEUIL_CALIBRE`) avant de lancer le lot final |

### Règle de calibration du seuil (écrite avant de lancer la calibration, et donc avant le lot final)

On fait varier le seuil sur la grille 0 ; 0,01 ; … ; 0,50. Pour chaque seuil, sur le lot de calibration, B s'abstient quand la confiance de M3 est inférieure au seuil, et l'**erreur** est la part de mauvais choix (`bon_choix` = 0, défini a posteriori : le modèle choisi n'est pas le plus précis des deux) **parmi les scénarios où B ne s'abstient pas**.

1. **Règle principale.** Seuil retenu = le plus petit seuil dont l'erreur, sur le lot de calibration **tous ratios réunis**, est $\le 10\,\%$.
2. **Repli.** Si aucun seuil de la grille ne convient (probable : à ratio 1,0 le choix est un pile ou face, ce qui alourdit l'erreur agrégée), on retient le seuil qui **minimise l'erreur agrégée sous la contrainte que l'abstention aux ratios < 1 reste $\le 30\,\%$** (à erreur égale, le plus petit seuil). Le notebook dit alors explicitement que le repli a servi.
3. La courbe erreur / abstention (agrégée et par ratio) est documentée, puis le seuil est figé.

Le lot final n'est lancé qu'après ce gel, et il n'a servi à aucun choix.

### Comptabilité de l'abstention (lot final)

Quand B s'abstient, il renvoie les deux modèles sans en désigner un. Pour comparer à M1 et à M3 sans abstention, on attribue à B une précision par scénario de deux façons :
- **« abstention = réponse au hasard »** (**mesure principale**) : B s'abstenant est traité comme un B qui répond un des deux modèles au hasard, donc précision = `hasard` du scénario. C'est la lecture neutre, comparable aux autres méthodes ;
- **« abstention = échec »** (borne pessimiste) : précision = 0 quand B s'abstient.

On rapporte aussi la précision de M3 **sur les seuls scénarios où il se prononce** et le **taux d'abstention**.

In [ ]:
RATIOS_T4 = (1.0, 0.85, 0.7, 0.5)
N_CALIB, N_FINAL = 10, 20                            # scénarios par ratio
GRAINES_RESERVEES = {"développement": range(0, 10), "diagnostic": range(100, 106),
                     "calibration": range(200, 240), "final": range(300, 380)}
CALIB_CSV, FINAL_CSV = "../results/t4_calibration.csv", "../results/t4_final.csv"

def graine_t4(lot, i_ratio, k):
    # calibration : 200 + 10*i_ratio + k ; final : 300 + 20*i_ratio + k  (i_ratio : indice dans RATIOS_T4, k : numéro du scénario)
    assert lot in ("calibration", "final")
    n, base = (N_CALIB, 200) if lot == "calibration" else (N_FINAL, 300)
    assert 0 <= i_ratio < len(RATIOS_T4) and 0 <= k < n
    return base + n * i_ratio + k

def plan_t4(lot):
    # (graine, ratio), k en boucle externe : une interruption laisse tous les ratios représentés
    n = N_CALIB if lot == "calibration" else N_FINAL
    return [(graine_t4(lot, i, k), r) for k in range(n) for i, r in enumerate(RATIOS_T4)]

# test : plages de graines disjointes, couverture exacte, aucune graine répétée
plages = list(GRAINES_RESERVEES.values())
assert all(set(a).isdisjoint(b) for i, a in enumerate(plages) for b in plages[i + 1:])
for lot, plage in (("calibration", GRAINES_RESERVEES["calibration"]), ("final", GRAINES_RESERVEES["final"])):
    graines = [g for g, _ in plan_t4(lot)]
    assert len(graines) == len(set(graines)) and sorted(graines) == list(plage)
print("graines T4 : plages disjointes ;", len(plan_t4("calibration")), "scénarios de calibration,", len(plan_t4("final")), "de lot final")

### Exécution des lots

`scenario_t4` réutilise `diag_scenario` : pour chaque scénario, M1, M2, M3, oracle, hasard, la confiance de M3, les comptes d'objets expliqués, les tailles, l'accord et les temps sont enregistrés **sans abstention** (`seuil = 0`). Le seuil s'applique ensuite sur ces valeurs, ce qui évite de relancer lincs pour tester un autre seuil.

Chaque scénario coûte environ 30 s. `RUN_CALIB = True` lance les 40 scénarios de calibration (environ 20 minutes), `RUN_FINAL = True` les 80 du lot final (environ 45 minutes). Les deux reprennent où ils s'étaient arrêtés si le CSV existe. Par défaut, les cellules relisent les CSV.

In [ ]:
def scenario_t4(lot, seed, ratio):
    return dict(lot=lot, **diag_scenario(seed, ratio))

def run_t4(lot, csv):
    try:
        rows = pd.read_csv(csv).to_dict("records")
    except FileNotFoundError:
        rows = []
    deja = {(int(r["graine"]), float(r["ratio"])) for r in rows}
    for seed, ratio in plan_t4(lot):
        if (seed, ratio) in deja:
            continue
        rows.append(scenario_t4(lot, seed, ratio))
        pd.DataFrame(rows).to_csv(csv, index=False)                       # sauvegarde au fil de l'eau
        r = rows[-1]
        print(f"[{lot}] {len(rows)}/{len(plan_t4(lot))} graine={seed} ratio={ratio}: M1={r['m1']:.3f} M2={r['m2']:.3f} "
              f"M3={r['m3']:.3f} oracle={r['oracle']:.3f} conf={r['conf_m3']:.3f} | {r['temps_pipeline']:.0f} s", flush=True)

RUN_CALIB = False
if RUN_CALIB:
    run_t4("calibration", CALIB_CSV)
calib = pd.read_csv(CALIB_CSV) if Path(CALIB_CSV).exists() else None
print("lot de calibration :", "absent" if calib is None else f"{len(calib)} scénarios, {calib.groupby('ratio').size().to_dict()}")

### Calibration du seuil

Les cellules suivantes appliquent la règle écrite plus haut au lot de calibration (et à lui seul).

In [ ]:
GRILLE_SEUILS = np.round(np.arange(0, 0.5001, 0.01), 2)
ERREUR_MAX, ABSTENTION_MAX = 0.10, 0.30              # règle principale / contrainte du repli

def _taux(df, choisi):
    # abstention et erreur (mauvais choix parmi les choix faits) pour un masque `choisi`
    return (1 - choisi.mean() if len(df) else np.nan,
            1 - df.bon_m3[choisi].mean() if choisi.any() else np.nan, int(choisi.sum()))

def courbe_seuil(df, seuils=GRILLE_SEUILS):
    # courbe erreur / abstention : agrégée (tous ratios), aux ratios < 1, puis par ratio
    rows = []
    for s in seuils:
        choisi = df.conf_m3 >= s
        ab, er, n = _taux(df, choisi)
        sous = df.ratio < 1
        ab1, er1, _ = _taux(df[sous], choisi[sous])
        row = dict(seuil=s, abstention=ab, erreur=er, n_choix=n, abstention_sous_1=ab1, erreur_sous_1=er1)
        for r in sorted(df.ratio.unique(), reverse=True):
            m = df.ratio == r
            row[f"abst_{r}"], row[f"err_{r}"], _ = _taux(df[m], choisi[m])
        rows.append(row)
    return pd.DataFrame(rows)

def choisir_seuil(courbe):
    ok = courbe[courbe.erreur <= ERREUR_MAX]                          # NaN (aucun choix) exclus automatiquement
    if len(ok):
        return float(ok.seuil.min()), "règle principale"
    ok = courbe[(courbe.abstention_sous_1 <= ABSTENTION_MAX) & courbe.erreur.notna()]
    if len(ok) == 0:
        return 0.0, "aucun seuil admissible : seuil 0 (pas d'abstention)"
    return float(ok[ok.erreur == ok.erreur.min()].seuil.min()), "repli : erreur minimale sous abstention (ratios < 1) <= 30 %"

# test de la règle sur des données synthétiques
fake = pd.DataFrame({"ratio": [0.5] * 6, "conf_m3": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5], "bon_m3": [0, 0, 1, 1, 1, 1]})
cf = courbe_seuil(fake)
assert choisir_seuil(cf) == (0.11, "règle principale")                # les deux mauvais choix (confiance 0 et 0,1) doivent s'abstenir : premier seuil de grille > 0,1
fake2 = pd.DataFrame({"ratio": [1.0, 1.0, 0.5, 0.5], "conf_m3": [0.3, 0.4, 0.3, 0.4], "bon_m3": [0, 0, 0, 0]})
assert choisir_seuil(courbe_seuil(fake2))[1].startswith("repli")
print("règle de calibration : tests ok")

In [ ]:
SEUIL_CALIBRE = 0.06            # FIGÉ d'après results/t4_calibration.csv (règle principale, voir ci-dessous), avant le lot final

if calib is not None and len(calib) == 4 * N_CALIB:
    courbe = courbe_seuil(calib)
    seuil_calcule, regle = choisir_seuil(courbe)
    print(f"seuil calculé sur le lot de calibration : {seuil_calcule:.2f} ({regle})")
    assert abs(seuil_calcule - SEUIL_CALIBRE) < 1e-9, "le lot de calibration ne redonne pas le seuil figé"
    cols = ["seuil", "abstention", "erreur", "n_choix", "abstention_sous_1", "erreur_sous_1"] + \
           [c for c in courbe.columns if c.startswith(("abst_", "err_"))]
    display(courbe[courbe.seuil.isin([0, 0.02, 0.04, 0.05, 0.06, 0.08, 0.1, 0.12, 0.15, 0.2, 0.3, 0.5])][cols].round(3))

### Seuil retenu : 0,06 (règle principale)

Sur le lot de calibration (40 scénarios, graines 200 à 239), la règle principale donne **`SEUIL_CALIBRE = 0,06`** : c'est le plus petit seuil dont l'erreur agrégée (4,0 %) passe sous 10 % (au seuil 0,05 elle vaut 11,1 %). Le repli n'a pas servi.

À lire avec prudence :
- **Le seuil est fragile.** Entre 0,05 et 0,06, l'erreur passe de 11,1 % (3 erreurs sur 27 choix) à 4,0 % (1 sur 25) : un seul scénario fait basculer la règle. Le lot de calibration est petit (10 par ratio).
- **L'erreur agrégée est en partie obtenue par l'abstention à ratio 1,0** : à 0,06, B s'abstient dans 7 scénarios sur 10 à ratio 1,0 et ne choisit que 3 fois (aucune erreur, par chance), alors qu'il se trompe 6 fois sur 10 sans abstention.
- **Aux ratios < 1, B s'abstient dans 26,7 % des scénarios** (sous la limite de 30 % du repli, qui ne servait pas ici) et fait 1 erreur sur 22 choix (4,5 %). Cette erreur est à ratio 0,85, où 1 choix sur 4 est faux (4 choix seulement).
- **Sans abstention, M3 se trompe 2 fois sur 30 aux ratios < 1** (ratios 0,85 et 0,7), alors que le diagnostic donnait 18 bons choix sur 18 : l'« absence d'erreur » du diagnostic n'était pas une propriété garantie.

Le seuil est figé : **le lot final est lancé avec 0,06, sans autre réglage.**

### Lot final

20 scénarios par ratio (graines 300 à 379), avec le seuil **figé** `SEUIL_CALIBRE`. `RUN_FINAL = True` lance les 80 scénarios (environ 45 minutes) ; par défaut la cellule relit `results/t4_final.csv`.

In [ ]:
RUN_FINAL = False
if RUN_FINAL:
    assert SEUIL_CALIBRE is not None, "calibrer et figer le seuil avant de lancer le lot final"
    run_t4("final", FINAL_CSV)
final = pd.read_csv(FINAL_CSV) if Path(FINAL_CSV).exists() else None
print("lot final :", "absent" if final is None else f"{len(final)} scénarios, {final.groupby('ratio').size().to_dict()}")

In [ ]:
def wilson(k, n, z=1.96):
    # intervalle de Wilson à 95 % d'une proportion k/n
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    demi = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (centre - demi, centre + demi)

def boot_ci(x, n_boot=1000, seed=0):
    # intervalle bootstrap (percentiles 2,5 / 97,5 %) de la moyenne, 1000 tirages avec remise des scénarios
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    moyennes = x[rng.integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return tuple(np.percentile(moyennes, [2.5, 97.5]))

def erreur_type(x):
    return float(np.std(x, ddof=1) / np.sqrt(len(x)))

def fmt_ic(m, lo, hi, nd=3):
    return f"{m:+.{nd}f} [{lo:+.{nd}f} ; {hi:+.{nd}f}]"

# tests des outils statistiques
lo, hi = wilson(18, 18); assert abs(lo - 0.824) < 0.005 and abs(hi - 1.0) < 1e-9          # borne basse de Wilson pour 18/18 : environ 0,82
assert wilson(0, 10)[0] < 1e-9 and 0.23 < wilson(5, 10)[0] < 0.24 and 0.76 < wilson(5, 10)[1] < 0.77      # 5/10 : [0,237 ; 0,763]
lo, hi = boot_ci(np.full(20, 0.3)); assert np.isclose(lo, 0.3) and np.isclose(hi, 0.3)
lo, hi = boot_ci(np.arange(20.)); assert lo < 9.5 < hi
print("outils statistiques : tests ok")

In [ ]:
NOMS_T4 = {"m1": "M1 modèle unique", "m2": "M2 taille", "m3": "M3 expliqués (sans abstention)",
           "b_hasard": "B avec abstention (= hasard)", "b_echec": "B avec abstention (= échec)",
           "oracle": "oracle", "hasard": "hasard"}
ORDRE_T4 = list(RATIOS_T4)

if final is not None and SEUIL_CALIBRE is not None:
    final = final.assign(choisi=final.conf_m3 >= SEUIL_CALIBRE)
    final["b_hasard"] = np.where(final.choisi, final.m3, final.hasard)           # abstention = réponse au hasard (principale)
    final["b_echec"] = np.where(final.choisi, final.m3, 0.0)                     # abstention = échec (borne pessimiste)
    g = final.groupby("ratio")
    print(f"Seuil figé : {SEUIL_CALIBRE:.2f} ; {len(final)} scénarios, {g.size().min()} à {g.size().max()} par ratio\n")

    # 1) précision en test : moyenne ± écart-type (erreur-type entre crochets)
    prec = pd.DataFrame({NOMS_T4[c]: g[c].apply(lambda s: f"{s.mean():.3f} ± {s.std():.3f} [{erreur_type(s):.3f}]")
                         for c in NOMS_T4}).loc[ORDRE_T4]
    print("Précision en test : moyenne ± écart-type [erreur-type]")
    display(prec.T)

    # 2) abstention de B et précision quand il se prononce
    ab = pd.DataFrame({
        "abstention": g.choisi.apply(lambda s: 1 - s.mean()),
        "n choix / n": g.choisi.apply(lambda s: f"{int(s.sum())}/{len(s)}"),
        "M3 quand B se prononce": g.apply(lambda d: d.m3[d.choisi].mean() if d.choisi.any() else np.nan),
        "bon choix quand B se prononce": g.apply(lambda d: d.bon_m3[d.choisi].mean() if d.choisi.any() else np.nan),
        "B (= hasard) − M3 sans abstention": g.apply(lambda d: (d.b_hasard - d.m3).mean())}).loc[ORDRE_T4]
    print("Abstention de B (seuil figé)")
    display(ab.round(3))

In [ ]:
if final is not None and SEUIL_CALIBRE is not None:
    # 3) différences appariées (même scénario) avec intervalle bootstrap à 95 %
    lignes = []
    for r in ORDRE_T4 + ["< 1"]:
        d = final[final.ratio == r] if r != "< 1" else final[final.ratio < 1]
        row = {"ratio": r, "n": len(d)}
        for nom, (a, b) in {"M3 − M1": ("m3", "m1"), "M3 − M2": ("m3", "m2"), "B(abst.=hasard) − M1": ("b_hasard", "m1")}.items():
            x = (d[a] - d[b]).to_numpy()
            row[nom] = fmt_ic(x.mean(), *boot_ci(x))
            row[nom + " (M3 gagne / égal / perd)" if nom == "M3 − M1" else nom + " gagne"] = \
                f"{int((x > 0).sum())} / {int((x == 0).sum())} / {int((x < 0).sum())}" if nom == "M3 − M1" else int((x > 0).sum())
        lignes.append(row)
    diff = pd.DataFrame(lignes).set_index("ratio")
    print("Différences appariées de précision : moyenne [IC bootstrap 95 %, 1000 tirages]")
    display(diff[["n", "M3 − M1", "M3 − M1 (M3 gagne / égal / perd)", "M3 − M2", "B(abst.=hasard) − M1"]])

    # 4) bon choix (sans abstention) avec intervalle de Wilson, partition, temps
    choix = []
    for r in ORDRE_T4 + ["< 1"]:
        d = final[final.ratio == r] if r != "< 1" else final[final.ratio < 1]
        k3, k2, n = int(d.bon_m3.sum()), int(d.bon_m2.sum()), len(d)
        l3, h3 = wilson(k3, n); l2, h2 = wilson(k2, n)
        choix.append({"ratio": r, "bon choix M3": f"{k3}/{n} = {k3 / n:.2f} [{l3:.2f} ; {h3:.2f}]",
                      "bon choix M2": f"{k2}/{n} = {k2 / n:.2f} [{l2:.2f} ; {h2:.2f}]",
                      "accord segments / sources": d.accord.mean(),
                      "taille trouvée (grand / petit)": f"{d.taille_grand.mean():.0f} / {d.taille_petit.mean():.0f}",
                      "taille réelle (A / malveillant)": f"{d.n_A.mean():.0f} / {d.n_M.mean():.0f}",
                      "temps M1 (s)": d.temps_m1.mean(), "temps pipeline (s)": d.temps_pipeline.mean()})
    print("Bon choix (sans abstention) avec intervalle de Wilson à 95 %, partitions et temps")
    display(pd.DataFrame(choix).set_index("ratio").round(2))

In [ ]:
import matplotlib.pyplot as plt

def figure_t4(final, chemin="../results/t4_final.png"):
    # précision en fonction du ratio : M1 (référence) et M3 (méthode de B) avec barres d'erreur à 95 %, oracle et hasard en repères
    surface, texte, texte2 = "#fcfcfb", "#0b0b0b", "#52514e"
    bleu, orange, gris = "#2a78d6", "#eb6834", "#8a8984"                 # emplacements 1 et 2 de la palette de référence
    g = final.groupby("ratio")
    x = np.arange(len(ORDRE_T4))
    fig, ax = plt.subplots(figsize=(8, 5.6), facecolor=surface)
    ax.set_facecolor(surface)
    def courbe(col, dx, **kw):
        m = np.array([g[col].mean()[r] for r in ORDRE_T4]); e = 1.96 * np.array([erreur_type(g[col].get_group(r)) for r in ORDRE_T4])
        ax.errorbar(x + dx, m, yerr=e if kw.pop("barres", True) else None, **kw)
        return m
    m3 = courbe("m3", 0.04, color=bleu, marker="o", ms=7, lw=2, capsize=4, label="M3 : pipeline à 2 modèles (méthode de B)")
    m1 = courbe("m1", -0.04, color=orange, marker="s", ms=7, lw=2, capsize=4, label="M1 : modèle unique (référence)")
    orc = courbe("oracle", 0, color=gris, marker="^", ms=6, lw=1.2, ls="--", barres=False, label="oracle : meilleur des 2 modèles appris")
    has = courbe("hasard", 0, color=gris, marker="x", ms=6, lw=1.2, ls=":", barres=False, label="hasard : moyenne des 2 modèles appris")
    haut, bas = sorted((m3[-1], m1[-1]), reverse=True)                    # la série la plus haute : étiquette au-dessus
    for v in (haut, bas):
        ax.annotate(f"{v:.2f}", (x[-1] + 0.12, v + (0.012 if v == haut else -0.03)), color=texte, fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([f"{r:g}".replace(".", ",") + f"\n(A = {100 / (1 + r):.0f} %)" for r in ORDRE_T4], color=texte2)
    ax.set_xlabel("Ratio du malveillant (objets du malveillant / objets de A)", color=texte2)
    ax.set_ylabel("Précision sur le test (modèle de A)", color=texte2)
    ax.set_ylim(0.45, 1.0); ax.tick_params(colors=texte2)
    ax.grid(axis="y", color="#e4e3df", lw=0.8); ax.set_axisbelow(True)
    for side in ("top", "right"): ax.spines[side].set_visible(False)
    for side in ("left", "bottom"): ax.spines[side].set_color("#c9c8c2")
    ax.set_title(f"B retrouve-t-il le modèle de A ?  {len(final)} scénarios, barres : ± 1,96 erreur-type", color=texte, fontsize=11, loc="left")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2, frameon=False, fontsize=8.5, labelcolor=texte2)
    fig.tight_layout()
    fig.savefig(chemin, dpi=160, facecolor=surface)
    return fig

if final is not None and SEUIL_CALIBRE is not None:
    figure_t4(final); plt.show()

### Résultats du lot final et lecture

*Valeurs de `results/t4_final.csv` : 80 scénarios, 20 par ratio, seuil figé à 0,06. Moyenne ± écart-type ; intervalles à 95 % (bootstrap pour les différences, Wilson pour les proportions).*

| Ratio | M1 modèle unique | M3 sans abstention | M2 taille | oracle | hasard | Bon choix M3 | Bon choix M2 | Abstention de B |
|---|---|---|---|---|---|---|---|---|
| 1,0 | 0,703 ± 0,144 | 0,675 ± 0,251 | 0,733 ± 0,219 | 0,853 ± 0,109 | 0,644 ± 0,052 | 9/20 = 0,45 [0,26 ; 0,66] | 12/20 | 60 % |
| 0,85 | 0,830 ± 0,094 | 0,862 ± 0,180 | 0,785 ± 0,203 | 0,900 ± 0,101 | 0,659 ± 0,056 | 18/20 = 0,90 [0,70 ; 0,97] | 15/20 | 50 % |
| 0,7 | 0,844 ± 0,072 | 0,872 ± 0,097 | 0,819 ± 0,162 | 0,880 ± 0,074 | 0,654 ± 0,046 | 19/20 = 0,95 [0,76 ; 0,99] | 17/20 | 15 % |
| 0,5 | 0,910 ± 0,041 | 0,867 ± 0,107 | 0,809 ± 0,179 | 0,867 ± 0,107 | 0,649 ± 0,055 | 20/20 = 1,00 [0,84 ; 1,00] | 17/20 | 10 % |

Différences appariées M3 − M1, moyenne [IC bootstrap 95 %] : ratio 1,0 : −0,028 [−0,123 ; +0,096] ; 0,85 : +0,032 [−0,047 ; +0,102] ; 0,7 : +0,028 [−0,019 ; +0,074] ; 0,5 : **−0,043 [−0,091 ; −0,003]** ; ratios < 1 réunis : +0,006 [−0,029 ; +0,036] (M3 gagne 35 scénarios sur 60).

**Ce qui est établi (20 scénarios par ratio, cette configuration)**
- **L'identification du modèle de A par les objets expliqués marche quand A est majoritaire** : 57 bons choix sur 60 aux ratios < 1 (0,95 [0,86 ; 0,98]), contre 49 sur 60 pour la taille (0,82 [0,70 ; 0,89]). À ratio 1,0 on obtient 9 sur 20 (0,45 [0,26 ; 0,66]), compatible avec le hasard : comme prévu, aucune méthode ne peut y choisir.
- **La séparation en deux modèles fonctionne à tous les ratios** : `oracle` vaut 0,85 à 0,90, y compris à ratio 1,0.

**Ce que le lot final contredit ou nuance par rapport au diagnostic**
- **M3 ne bat pas M1 en précision.** Sur les ratios < 1 la différence moyenne est de +0,6 point, indiscernable de zéro, et à ratio 0,5 **M1 est meilleur** de 4,3 points (IC [−9,1 ; −0,3]). Le diagnostic (6 scénarios par ratio) avait suggéré +5 à +11 points à 0,7 et 0,85 : c'était dans le bruit. M1 est en outre environ 7 fois plus rapide (4,2 s contre 27 à 31 s).
- **Le gain possible n'est pas réalisé** : à ratio 0,85 et 0,7, `oracle` dépasse M1 de 7,0 et 3,7 points, mais B n'atteint pas l'`oracle` (mauvais choix et segments impurs sont les explications probables, non isolées). À ratio 0,5, `oracle` est même **sous** M1 (−4,3 points) : apprendre sur un segment impur (accord d'environ 0,72) coûte peut-être plus que ce que la séparation apporte (explication probable, non testée).
- **Les « 18 sur 18 » du diagnostic n'étaient pas une garantie** : 3 erreurs sur 60 ici.

**L'abstention (seuil 0,06)**
- B s'abstient dans 10 %, 15 %, 50 % et 60 % des scénarios aux ratios 0,5 ; 0,7 ; 0,85 ; 1,0.
- **Elle ne fait pas gagner de précision** : en comptant l'abstention comme une réponse au hasard (mesure principale), B est en dessous de M3 sans abstention de 0,7 à 9,1 points. À ratio 0,85 elle écarte 8 choix corrects sur 10 abstentions.
- **L'objectif de calibration n'est pas tenu sur le lot final** : l'erreur parmi les scénarios où B se prononce vaut **6/53 = 11,3 % [5,3 ; 22,6]**, pour une cible de 10 %. Elle est de 1/45 (2,2 %) aux ratios < 1 et de **5/8 à ratio 1,0**, où la confiance (0,046 en moyenne) ne sépare pas assez bien le cas indécidable des cas décidables (0,07 à ratio 0,85, 0,15 à 0,7, 0,16 à 0,5).

**Conclusion sur l'approche.** Ce lot n'établit pas que le pipeline à deux modèles apporte un gain de précision sur le modèle unique dans cette configuration. Ce qu'il apporte est structurel : il identifie A (95 % de bons choix quand A est majoritaire), apprend aussi le modèle du malveillant et signale le cas indécidable, ce que M1 ne fait pas. Si seule la précision sur le modèle de A compte, M1 suffit et coûte 7 fois moins.

**Limites.** Une seule configuration (6 critères, 3 catégories, 300 objets de A) ; « bon choix » défini a posteriori ; lincs s'arrête sur un temps (non reproductible à l'octet près) ; calibration sur 40 scénarios seulement (seuil fragile) ; M1 n'est évalué que par rapport au modèle de A (à ratio 1,0 son 0,70 est un modèle de compromis, non une identification) ; pas de correction pour les comparaisons multiples ; aucun ratio entre 0,85 et 1,0, donc on ne sait pas où l'identification cesse exactement.

## Améliorations de B

Le lot final de T4 montre que M3 identifie A (95 % de bons choix pour un ratio < 1) mais **ne bat pas M1**, le modèle unique appris sur tout le mélange : l'écart moyen est de +0,6 point, indiscernable de zéro. On teste donc si une amélioration de B donne un gain de précision **mesurable** sur le modèle de A.

| Méthode | Principe | Rôle |
|---|---|---|
| **plafond** | lincs appris sur les **seuls objets de A** (la source est connue), même budget que M1 | borne supérieure : B ne peut pas le faire |
| **M1_eq** | M1 avec le **budget de calcul du pipeline** (`POLISH_SECONDS` × 7 = 28 s par apprentissage, soit le temps mural du pipeline) | comparaison équitable : M1 ne dispose que de 4 s |
| **M1_trim, `trim_e2`** | on apprend M1, on retire les objets dont l'écart de catégorie $\lvert h_{prédit} - h_{reçu} \rvert \ge 2$, on réapprend, 3 tours au plus | modèle unique **robuste** : les objets du malveillant sont du bruit pour A |
| **M1_trim, `trim_f40`** | même chose, mais on retire tous les mal classés, au plus 40 % des objets, les pires d'abord (écart puis marge) | variante plus agressive |
| **M3_refit, `refit_exp`** | après le choix de M3, on réapprend un modèle (`POLISH_SECONDS` × 2) sur les objets que le modèle choisi **explique** | corrige la partition impure en s'appuyant sur l'identification |
| **M3_refit, `refit_union`** | idem, sur les objets expliqués **union** ceux du segment choisi | variante moins sélective |

Références conservées : M1 (4 s), M3, oracle et hasard du pipeline. Aucune variante n'utilise la vérité terrain, sauf le plafond.

**Détails de conception**
- *M1_trim :* à chaque tour, la sélection se fait sur le mélange **entier** avec le modèle courant (pas de retrait cumulatif). On s'arrête si la sélection ne change plus ou si on garderait moins de 50 % des objets : en dessous, le modèle n'a plus rien à apprendre.
- *M3_refit :* garde-fou comme dans le polish du jalon 2 : si le modèle réappris fait moins bien que le modèle choisi sur ses données d'apprentissage, on garde ce dernier.
- *Budget de M1_eq :* le pipeline dure 27 à 31 s (mesuré en T4) ; sept fois 4 s donne 28 s pour M1, avec les mêmes 4 apprentissages en parallèle.

**Protocole** (graines disjointes de toutes les plages déjà utilisées : 0-9, 100-105, 200-239, 300-379, 400-419) :
1. **Exploration** : graines 500 à 515, ratios {0,85 ; 0,5}, 8 scénarios par ratio, **toutes** les variantes.
2. **Règle de sélection, écrite avant de regarder les résultats :** parmi les quatre variantes de B (hors plafond et hors M1_eq, qui sont des références), on garde les **deux de plus grande précision moyenne sur l'ensemble de l'exploration** (16 scénarios, les deux ratios réunis).
3. **Confirmation** : graines 600 à 699, ratios {1,0 ; 0,925 ; 0,85 ; 0,7 ; 0,5}, 20 scénarios par ratio, avec les deux variantes retenues, M1, M1_eq, M3, plafond, oracle et hasard. Mêmes scénarios pour toutes les méthodes, et le pipeline n'est lancé qu'une fois par scénario.

Le calcul est séquentiel : lincs est limité par le temps et multithread, donc deux calculs simultanés fausseraient les résultats.

In [ ]:
FACTEUR_M1_EQ = 7.0          # M1_eq : POLISH_SECONDS x 7 = 28 s par apprentissage, soit le temps mural du pipeline (27 à 31 s)
ROUNDS_TRIM   = 3            # M1_trim : nombre maximal de tours « retirer puis réapprendre »
FRAC_MAX_TRIM = 0.40         # M1_trim (variante frac40) : fraction maximale d'objets retirés
FACTEUR_REFIT = 2.0          # M3_refit : POLISH_SECONDS x 2 pour le réapprentissage final
VARIANTES_B   = ("trim_e2", "trim_f40", "refit_exp", "refit_union")      # variantes explorées (le plafond et M1_eq sont des références)

def learn_ceiling(X, y, source, p, seed=0):
    # PLAFOND : lincs appris sur les seuls objets de A (la source est connue). B ne peut pas le faire : borne supérieure.
    A = np.asarray(source) == 0
    return learn_single(X[A], y[A], p, seed=seed)

def _fit_eq(X, y, p, seed, seconds):
    # patience quasi illimitée : par défaut lincs s'arrête après 30 itérations sans progrès (M1_eq durait 14 s au lieu de 28 s)
    m = learn_mrsort(X, y, p, seed=seed, seconds=seconds, patience=10**6)
    return m, float((predict(m, X) == y).mean())

def learn_single_eq(X, y, p, seed=0, n_starts=4):
    # M1_eq : même modèle unique que M1 (4 départs parallèles, meilleure précision sur le mélange reçu), mais avec le budget
    # de calcul du pipeline : le temps est borné par POLISH_SECONDS x FACTEUR_M1_EQ, pas par la patience (comparaison équitable)
    t = time.time()
    seconds = budget(POLISH_SECONDS * FACTEUR_M1_EQ, X.shape[1], p)
    runs = Parallel(n_jobs=n_starts, backend="loky")(delayed(_fit_eq)(X, y, p, seed * 100 + r, seconds) for r in range(n_starts))
    model, acc = max(runs, key=lambda r: r[1])
    return dict(model=model, train_accuracy=acc, accuracies=[a for _, a in runs], temps=time.time() - t)

def conserves(model, X, y, mode, frac_max=FRAC_MAX_TRIM):
    # masque des objets conservés par M1_trim. « ecart2 » : on retire les objets à écart de catégorie >= 2.
    # « frac40 » : on retire les mal classés, au plus frac_max * N, les pires d'abord (écart, puis marge).
    ecart = np.abs(predict(model, X) - y)
    if mode == "ecart2":
        return ecart < 2
    keep = np.ones(len(y), bool)
    mal = np.where(ecart > 0)[0]
    n_max = int(frac_max * len(y))
    if len(mal) > n_max:
        marge = margins(model, X, y)
        mal = mal[np.lexsort((marge[mal], -ecart[mal]))[:n_max]]          # clé principale : écart décroissant ; puis marge croissante
    keep[mal] = False
    return keep

def learn_single_trim(X, y, p, single, mode, seed=0, rounds=ROUNDS_TRIM, min_conserve=0.5):
    # M1_trim : à partir de M1 (`single`), on retire les objets douteux (selon le modèle courant, sur le mélange entier),
    # on réapprend sur les objets conservés, et on répète. Arrêt si la sélection ne change plus ou si on garderait < 50 %.
    model, keep, tours = single["model"], np.ones(len(y), bool), 0
    for r in range(rounds):
        nouveau = conserves(model, X, y, mode)
        if nouveau.sum() < max(10, min_conserve * len(y)) or (nouveau == keep).all():
            break
        keep = nouveau
        model = learn_single(X[keep], y[keep], p, seed=seed + 1000 * (r + 1))["model"]
        tours += 1
    return dict(model=model, part_conservee=float(keep.mean()), tours=tours)

def refit_after_identification(X, y, p, best, decision, mode, seed=0):
    # M3_refit : après le choix de M3, on réapprend un modèle (polish long) sur les objets que le modèle choisi explique
    # (mode « exp »), ou sur ces objets union ceux du segment choisi (mode « union »). Garde-fou comme au polish :
    # si le réapprentissage ne fait pas au moins aussi bien que le modèle choisi sur ses données, on garde ce dernier.
    k, model = decision.indice_segment_A, decision.modele_A
    expl = predict(model, X) == y
    idx = expl if mode == "exp" else (expl | (best["seg"] == k))
    if idx.sum() < 10:
        return dict(model=model, n_train=int(idx.sum()), refait=False)
    new = learn_single(X[idx], y[idx], p, seed=seed, polish_seconds=POLISH_SECONDS * FACTEUR_REFIT)["model"]
    mieux = (predict(new, X[idx]) == y[idx]).mean() >= (predict(model, X[idx]) == y[idx]).mean()
    return dict(model=new if mieux else model, n_train=int(idx.sum()), refait=bool(mieux))

def b_scenario(seed, ratio, variantes=VARIANTES_B):
    # un scénario : M1, plafond, M1_eq, pipeline (M3, oracle, hasard) et les variantes demandées, sur LE MÊME scénario
    sc = make_scenario(ScenarioConfig(ratio_malveillant=ratio), seed)
    p = sc.config.p
    acc = lambda mod: model_accuracy(mod, sc.X_test, sc.y_test)
    out = dict(graine=seed, ratio=ratio, n_A=int((sc.source == 0).sum()), n_M=int((sc.source == 1).sum()))

    t = time.time(); single = learn_single(sc.X, sc.y, p, seed=seed); out.update(m1=acc(single["model"]), temps_m1=time.time() - t)
    t = time.time(); ceil = learn_ceiling(sc.X, sc.y, sc.source, p, seed=seed); out.update(plafond=acc(ceil["model"]), temps_plafond=time.time() - t)
    t = time.time(); eq = learn_single_eq(sc.X, sc.y, p, seed=seed); out.update(m1_eq=acc(eq["model"]), temps_m1_eq=time.time() - t)

    t = time.time(); best, _ = learn_B(sc.X, sc.y, p, seed=seed); out["temps_pipeline"] = time.time() - t
    d3 = identify_A(best, sc.X, sc.y, seuil=0.0)                               # M3 : objets expliqués, sans abstention
    accs = [acc(mod) for mod in best["models"]]
    out.update(m3=accs[d3.indice_segment_A], oracle=max(accs), hasard=float(np.mean(accs)),
               bon_m3=float(accs[d3.indice_segment_A] >= max(accs)), conf_m3=d3.confiance,
               accord=float(max((best["seg"] == sc.source).mean(), (best["seg"] != sc.source).mean())))

    for v in variantes:
        t = time.time()
        if v.startswith("trim_"):
            r = learn_single_trim(sc.X, sc.y, p, single, "ecart2" if v == "trim_e2" else "frac40", seed=seed)
            out.update({f"part_{v}": r["part_conservee"], f"tours_{v}": r["tours"]})
        else:
            r = refit_after_identification(sc.X, sc.y, p, best, d3, "exp" if v == "refit_exp" else "union", seed=seed)
            out.update({f"ntrain_{v}": r["n_train"], f"refait_{v}": float(r["refait"])})
        out.update({v: acc(r["model"]), f"temps_{v}": time.time() - t})
    return out

In [ ]:
GRAINES_RESERVEES.update({"ratio 0,925": range(400, 420), "exploration B": range(500, 516), "confirmation B": range(600, 700)})
RATIOS_EXPLO, N_EXPLO = (0.85, 0.5), 8                  # exploration : 2 ratios x 8 scénarios
RATIOS_CONF,  N_CONF  = (1.0, 0.925, 0.85, 0.7, 0.5), 20   # confirmation : 5 ratios x 20 scénarios
EXPLO_CSV, CONF_CSV = "../results/b_explore.csv", "../results/b_confirm.csv"

def plan_b(phase):
    # (graine, ratio), k en boucle externe : une interruption laisse tous les ratios représentés
    ratios, n, base = (RATIOS_EXPLO, N_EXPLO, 500) if phase == "exploration" else (RATIOS_CONF, N_CONF, 600)
    return [(base + n * i + k, r) for k in range(n) for i, r in enumerate(ratios)]

# test : plages disjointes (y compris les nouvelles), couverture exacte, aucune graine répétée
plages = list(GRAINES_RESERVEES.values())
assert all(set(a).isdisjoint(b) for i, a in enumerate(plages) for b in plages[i + 1:])
for phase, nom in (("exploration", "exploration B"), ("confirmation", "confirmation B")):
    graines = [g for g, _ in plan_b(phase)]
    assert len(graines) == len(set(graines)) and sorted(graines) == list(GRAINES_RESERVEES[nom])
print("graines B :", len(plan_b("exploration")), "scénarios d'exploration,", len(plan_b("confirmation")), "de confirmation ; plages disjointes")

def choisir_variantes(explo, k=2):
    # RÈGLE DE SÉLECTION, écrite avant de regarder les résultats d'exploration : parmi les variantes de B (hors plafond et hors
    # M1_eq, qui sont des références), on garde les k variantes de plus grande précision moyenne sur l'ensemble de l'exploration.
    moyennes = explo[list(VARIANTES_B)].mean().sort_values(ascending=False)
    return list(moyennes.index[:k]), moyennes

# test de la règle sur des données synthétiques
fake = pd.DataFrame({"trim_e2": [0.7, 0.7], "trim_f40": [0.9, 0.8], "refit_exp": [0.85, 0.85], "refit_union": [0.6, 0.6]})
assert choisir_variantes(fake)[0] == ["trim_f40", "refit_exp"]

def run_b(phase, csv, variantes):
    try:
        rows = pd.read_csv(csv).to_dict("records")
    except FileNotFoundError:
        rows = []
    deja = {(int(r["graine"]), float(r["ratio"])) for r in rows}
    for seed, ratio in plan_b(phase):
        if (seed, ratio) in deja:
            continue
        rows.append(dict(phase=phase, **b_scenario(seed, ratio, variantes)))
        pd.DataFrame(rows).to_csv(csv, index=False)                       # sauvegarde au fil de l'eau
        r = rows[-1]
        print(f"[{phase}] {len(rows)}/{len(plan_b(phase))} graine={seed} ratio={ratio}: M1={r['m1']:.3f} M1_eq={r['m1_eq']:.3f} "
              f"M3={r['m3']:.3f} plafond={r['plafond']:.3f} " + " ".join(f"{v}={r[v]:.3f}" for v in variantes)
              + f" | {r['temps_pipeline']:.0f} s", flush=True)

RUN_EXPLO = False            # True : lance l'exploration (environ 30 minutes)
RUN_CONF  = False            # True : lance la confirmation (environ 2 h), avec les variantes retenues par la règle ci-dessus
if RUN_EXPLO:
    run_b("exploration", EXPLO_CSV, VARIANTES_B)
explo = pd.read_csv(EXPLO_CSV) if Path(EXPLO_CSV).exists() else None
RETENUES = choisir_variantes(explo)[0] if explo is not None and len(explo) == len(plan_b("exploration")) else None
print("exploration :", "absente" if explo is None else f"{len(explo)} scénarios", "| variantes retenues :", RETENUES)
if RUN_CONF and RETENUES:
    run_b("confirmation", CONF_CSV, RETENUES)
conf = pd.read_csv(CONF_CSV) if Path(CONF_CSV).exists() else None
print("confirmation :", "absente" if conf is None else f"{len(conf)} scénarios")

In [ ]:
# tests légers (sans benchmark) des fonctions de la section « Améliorations de B »
rng_t = np.random.default_rng(5)
mod_t = random_model(6, 3, rng_t)
X_t = rng_t.random((400, 6)); y_t = predict(mod_t, X_t)
bruit = rng_t.random(400) < 0.3                                           # 30 % d'étiquettes remplacées au hasard
y_b = np.where(bruit, rng_t.integers(0, 3, 400), y_t)
mal = predict(mod_t, X_t) != y_b

# 1. conserves : « ecart2 » ne retire que des écarts >= 2 ; « frac40 » ne retire que des mal classés, au plus 40 % ; la plus grande fraction demandée retire les pires
k2 = conserves(mod_t, X_t, y_b, "ecart2")
assert (~k2).sum() == (np.abs(predict(mod_t, X_t) - y_b) >= 2).sum() and (~k2).sum() > 0
k4 = conserves(mod_t, X_t, y_b, "frac40")
assert (~k4).sum() <= 0.4 * 400 and mal[~k4].all()
petit = conserves(mod_t, X_t, y_b, "frac40", frac_max=0.02)
assert (~petit).sum() == 8 and np.abs(predict(mod_t, X_t) - y_b)[~petit].min() >= np.abs(predict(mod_t, X_t) - y_b)[mal & petit].max()
assert conserves(mod_t, X_t, y_t, "frac40").all() and conserves(mod_t, X_t, y_t, "ecart2").all()     # sans bruit : rien n'est retiré

# 2. avec lincs, sur un petit scénario : M1_trim et M3_refit renvoient un modèle valide et ne plantent pas
sc_b = make_scenario(ScenarioConfig(n_A=100, ratio_malveillant=0.5), seed=21)
s1 = learn_single(sc_b.X, sc_b.y, 3, seed=7, polish_seconds=0.5)
for mode in ("ecart2", "frac40"):
    tr = learn_single_trim(sc_b.X, sc_b.y, 3, s1, mode, seed=7)
    assert tr["model"]["profiles"].shape == (2, 6) and 0.5 <= tr["part_conservee"] <= 1 and 0 <= tr["tours"] <= ROUNDS_TRIM
best_b = perfect_result(sc_b)                                              # séparation parfaite : le choix est le bon
d_b = identify_A(best_b, sc_b.X, sc_b.y, seuil=0.0)
for mode in ("exp", "union"):
    rf = refit_after_identification(sc_b.X, sc_b.y, 3, best_b, d_b, mode, seed=7)
    assert rf["model"]["profiles"].shape == (2, 6) and rf["n_train"] >= 100
pl = learn_ceiling(sc_b.X, sc_b.y, sc_b.source, 3, seed=7)
assert (predict(pl["model"], sc_b.X[sc_b.source == 0]) == sc_b.y[sc_b.source == 0]).mean() > 0.6      # appris sur A seul
print("tests des améliorations de B : ok")

In [ ]:
import matplotlib.pyplot as plt

NOMS_B = {"m1": "M1 (4 s)", "m1_eq": "M1_eq (28 s)", "m3": "M3 pipeline", "pipeline": "pipeline (learn_B)", "plafond": "plafond (A seul)", "oracle": "oracle", "hasard": "hasard",
          "trim_e2": "trim_e2", "trim_f40": "trim_f40", "refit_exp": "refit_exp", "refit_union": "refit_union"}

def ordre_ratios(df):
    return sorted(df.ratio.unique(), reverse=True)

def table_precision(df, cols):
    # moyenne ± écart-type par ratio, et sur l'ensemble des ratios < 1
    ordre = ordre_ratios(df)
    lignes = {}
    for r in ordre:
        d = df[df.ratio == r]
        lignes[f"{r:g}".replace(".", ",")] = {NOMS_B[c]: f"{d[c].mean():.3f} ± {d[c].std(ddof=1):.3f}" for c in cols}
    d = df[df.ratio < 1]
    lignes["< 1 réunis"] = {NOMS_B[c]: f"{d[c].mean():.3f} ± {d[c].std(ddof=1):.3f}" for c in cols}
    return pd.DataFrame(lignes).T

def table_differences(df, variante, contre):
    # différences appariées (même scénario) variante - contre : moyenne [IC bootstrap 95 %] et victoires / égalités / défaites
    lignes = {}
    groupes = [(f"{r:g}".replace(".", ","), df[df.ratio == r]) for r in ordre_ratios(df)] + [("< 1 réunis", df[df.ratio < 1])]
    for nom, d in groupes:
        row = {}
        for c in contre:
            x = (d[variante] - d[c]).to_numpy()
            lo, hi = boot_ci(x)
            row[f"{NOMS_B[variante]} − {NOMS_B[c]}"] = f"{fmt_ic(x.mean(), lo, hi)}  ({int((x > 0).sum())}/{int((x == 0).sum())}/{int((x < 0).sum())})"
        lignes[nom] = row
    return pd.DataFrame(lignes).T

def part_du_chemin(df, variantes):
    # (précision de la variante - M1) / (plafond - M1), sur les moyennes par ratio : part de l'écart M1 -> plafond comblée
    lignes = {}
    for r in ordre_ratios(df) + ["< 1"]:
        d = df[df.ratio < 1] if r == "< 1" else df[df.ratio == r]
        ecart = d.plafond.mean() - d.m1.mean()
        lignes["< 1 réunis" if r == "< 1" else f"{r:g}".replace(".", ",")] = dict(
            {"plafond − M1": f"{ecart:+.3f}"}, **{NOMS_B[c]: f"{(d[c].mean() - d.m1.mean()) / ecart:+.0%}" for c in variantes})
    return pd.DataFrame(lignes).T

def table_temps(df, cols):
    return pd.DataFrame({NOMS_B[c]: [df[f"temps_{c}"].mean()] for c in cols if f"temps_{c}" in df}, index=["temps moyen (s)"]).round(1)

def figure_b(conf, variante, chemin="../results/b_confirm.png"):
    # précision selon le ratio : M1, M1_eq, meilleure variante de B, M3 (barres : ± 1,96 erreur-type) et plafond en repère
    surface, texte, texte2 = "#fcfcfb", "#0b0b0b", "#52514e"
    gris = "#8a8984"
    series = [("m1", "M1 (4 s)", "#2a78d6", "o", -0.12), ("m1_eq", "M1_eq (28 s)", "#eb6834", "s", -0.04),
              (variante, f"{variante} (B amélioré)", "#1baf7a", "D", 0.04), ("m3", "M3 pipeline", "#eda100", "^", 0.12)]
    ordre = ordre_ratios(conf)
    x = np.arange(len(ordre))
    g = conf.groupby("ratio")
    fig, ax = plt.subplots(figsize=(8.4, 5.8), facecolor=surface)
    ax.set_facecolor(surface)
    for col, nom, couleur, marqueur, dx in series:
        m = np.array([g[col].mean()[r] for r in ordre])
        e = 1.96 * np.array([erreur_type(g[col].get_group(r)) for r in ordre])
        ax.errorbar(x + dx, m, yerr=e, color=couleur, marker=marqueur, ms=6, lw=1.8, capsize=3, label=nom)
    pl = np.array([g["plafond"].mean()[r] for r in ordre])
    ax.plot(x, pl, color=gris, marker="x", ms=6, lw=1.2, ls="--", label="plafond : lincs sur A seul (non réalisable)")
    ax.set_xlim(-0.4, len(ordre) - 1 + 0.4)
    ax.set_xticks(x); ax.set_xticklabels([f"{r:g}".replace(".", ",") + f"\n(A = {100 / (1 + r):.0f} %)" for r in ordre], color=texte2)
    ax.set_xlabel("Ratio du malveillant (objets du malveillant / objets de A)", color=texte2)
    ax.set_ylabel("Précision sur le test (modèle de A)", color=texte2)
    ax.set_ylim(0.5, 1.0); ax.tick_params(colors=texte2)
    ax.grid(axis="y", color="#e4e3df", lw=0.8); ax.set_axisbelow(True)
    for side in ("top", "right"): ax.spines[side].set_visible(False)
    for side in ("left", "bottom"): ax.spines[side].set_color("#c9c8c2")
    ax.set_title(f"Une amélioration de B rattrape-t-elle le plafond ?  {len(conf)} scénarios, barres : ± 1,96 erreur-type", color=texte, fontsize=11, loc="left")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False, fontsize=8.5, labelcolor=texte2)
    fig.tight_layout()
    fig.savefig(chemin, dpi=160, facecolor=surface)
    return fig

def analyse_confirmation(conf, retenues):
    # tous les tableaux de la confirmation (appelée quand les 100 scénarios sont terminés)
    refs = ["m1", "m1_eq", "m3", "plafond", "oracle", "hasard"]
    print("— Précision (moyenne ± écart-type)"); display(table_precision(conf, refs + list(retenues)))
    for v in list(retenues) + ["m3"]:
        print(f"— Différences appariées : {NOMS_B[v]} (moyenne [IC bootstrap 95 %]  (gagne/égal/perd))")
        display(table_differences(conf, v, [c for c in ["m1", "m1_eq", "m3"] if c != v]))
    print("— Part du chemin M1 → plafond comblée"); display(part_du_chemin(conf, ["m1_eq", "m3"] + list(retenues)))
    print("— Temps"); display(table_temps(conf, ["m1", "plafond", "m1_eq", "pipeline"] + list(retenues)))
    print("— Bon choix de M3 (Wilson 95 %) et précision des variantes de réapprentissage selon ce choix")
    k = conf.groupby("ratio").bon_m3.sum().astype(int); n = conf.groupby("ratio").size()
    lignes = {f"{r:g}".replace(".", ","): {"bon choix M3": f"{k[r]}/{n[r]} = {k[r] / n[r]:.2f} [{wilson(k[r], n[r])[0]:.2f} ; {wilson(k[r], n[r])[1]:.2f}]"} for r in ordre_ratios(conf)}
    display(pd.DataFrame(lignes).T)
    refit = [v for v in retenues if v.startswith("refit_")]
    for v in refit:
        print(f"{NOMS_B[v]} selon que M3 a bien choisi :")
        display(conf.groupby("bon_m3")[["m3", v, "m1"]].agg(["mean", "count"]).round(3))

In [ ]:
if conf is not None and RETENUES is not None and len(conf) == len(plan_b("confirmation")):
    analyse_confirmation(conf, RETENUES)
    meilleure = max(RETENUES, key=lambda v: conf[v].mean())
    figure_b(conf, meilleure); plt.show()
else:
    print("confirmation incomplète ou variantes non retenues : lancer RUN_EXPLO puis RUN_CONF")

### Résultats de l'exploration et de la confirmation, et lecture

**Exploration** (16 scénarios, graines 500 à 515, ratios 0,85 et 0,5) : précision moyenne de `trim_e2` 0,866 ; `trim_f40` 0,852 ; `refit_exp` 0,872 ; `refit_union` 0,870. La règle écrite à l'avance retient **`refit_exp` et `refit_union`** (`trim_e2` est troisième, à 0,6 point). Sur ces 16 scénarios, `trim_e2` dépassait M1 de 1,2 point à 0,85 et de 1,7 point à 0,5 : trop peu de scénarios pour conclure, et la règle l'exclut de la confirmation.

**Confirmation** (100 scénarios, graines 600 à 699, 20 par ratio). Précision sur le test (modèle de A), moyenne sur 20 scénarios (écarts-types dans les tableaux ci-dessus) :

| Ratio | M1 (4 s) | M1_eq (28 s) | M3 | refit_exp | refit_union | plafond | oracle |
|---|---|---|---|---|---|---|---|
| 1,0 | 0,694 | 0,695 | 0,671 | 0,677 | 0,677 | 0,980 | 0,852 |
| 0,925 | 0,737 | 0,745 | 0,728 | 0,734 | 0,733 | 0,976 | 0,855 |
| 0,85 | 0,791 | 0,793 | 0,847 | 0,848 | 0,850 | 0,982 | 0,880 |
| 0,7 | 0,846 | 0,856 | 0,796 | 0,807 | 0,809 | 0,980 | 0,839 |
| 0,5 | 0,920 | 0,913 | 0,909 | 0,914 | 0,912 | 0,978 | 0,909 |
| < 1 réunis | 0,823 | 0,827 | 0,820 | 0,826 | 0,826 | 0,979 | 0,871 |

**Établi (100 scénarios, cette configuration)**
- **Le plafond est élevé et stable : 0,98 à tous les ratios.** Quand lincs n'apprend que les objets de A, il retrouve presque parfaitement leur modèle. L'écart de B au plafond est donc entièrement dû à la contamination par le malveillant : 0,29 à ratio 1,0, 0,24 à 0,925, 0,19 à 0,85, 0,14 à 0,7 et 0,06 à 0,5 pour M1.
- **Donner plus de calcul à M1 ne change rien** : M1_eq − M1 = +0,003 (bootstrap [−0,008 ; +0,014]) sur les 100 scénarios. Le budget de 4 s n'est pas ce qui limite M1.
- **Aucune amélioration ne bat M1 de façon mesurable** : `refit_exp` − M1 vaut +0,002 [−0,039 ; +0,042] aux ratios < 1, `refit_union` − M1 vaut +0,003 [−0,038 ; +0,043] ; aucun ratio n'a un intervalle qui exclut zéro.
- **Le réapprentissage améliore très légèrement M3** : `refit_exp` − M3 = +0,006 [+0,003 ; +0,009] et `refit_union` − M3 = +0,006 [+0,003 ; +0,009] aux ratios < 1 (0,6 point), soit environ 2 % de l'écart M3 → plafond. L'effet est réel mais négligeable.
- **La part du chemin de M1 vers le plafond comblée est de +2 % aux ratios < 1** pour les deux variantes (de −6 % à +31 % selon le ratio, avec un signe qui change : bruit).

**Indicatif**
- **Les signes par ratio ne sont pas stables.** M3 − M1 vaut +0,057 à 0,85 et −0,050 à 0,7, alors que le lot de T4 donnait +0,032 et +0,028 aux mêmes ratios. Avec 20 scénarios par ratio et des écarts-types de 0,16 à 0,26 pour M3, seule la moyenne sur plusieurs ratios dit quelque chose : −0,003 [−0,045 ; +0,037] (aux ratios < 1).
- **Les erreurs d'identification annulent les gains.** Aux ratios < 1, quand M3 choisit le bon modèle (68 scénarios sur 80), il fait 0,880 contre 0,830 pour M1 (+5 points) ; quand il se trompe (12 scénarios), il fait 0,480 contre 0,789 (−31 points). Conditionner sur le résultat favorise le « bon choix », donc cette décomposition est descriptive, pas une estimation d'effet.
- **Bon choix de M3 : 12/20, 14/20, 17/20, 17/20 et 20/20** pour les ratios 1,0 ; 0,925 ; 0,85 ; 0,7 ; 0,5. Sur les ratios 0,85 ; 0,7 et 0,5, 54 sur 60 (0,90) contre 57 sur 60 en T4 : cohérent, avec une variabilité à garder en tête.

**Non conclu**
- **Pourquoi le réapprentissage n'apporte presque rien.** Hypothèse non testée : il s'entraîne uniquement sur les objets que le modèle choisi explique déjà. Les objets de A que ce modèle classe mal sont écartés, et le réapprentissage ne peut donc pas les corriger (il se confirme lui-même). L'écart restant (0,98 du plafond contre 0,82) est précisément fait de ces erreurs.
- **`trim_e2` et `trim_f40` n'ont pas été confirmés** : la règle de sélection les a écartés après l'exploration. On ne sait pas s'ils auraient fait mieux.

**Limites** : 20 scénarios par ratio, une seule configuration (6 critères, 3 catégories, 300 objets de A), exploration sur 16 scénarios seulement, plafond non réalisable en pratique, lincs non reproductible à l'octet près (une seule exécution).